# 6. Experimental Configuration and Data Pipeline

## Notebook Purpose

This notebook prepares, verifies, and runs the experimental pipeline for
classifying ECG signals with a 1D CNN.

The workflow is:

1. load the non-normalized signals and the existing data splits;
2. adapt signals, labels, and indices to the PyTorch format;
3. build the DataLoaders and verify the data partitions;
4. define the initial 1D CNN architecture;
5. configure the loss function, optimizer, and early stopping;
6. train and evaluate the initial architecture on the five folds;
7. analyze its results at different decision thresholds;
8. use Optuna to search for a hyperparameter configuration;
9. evaluate the best configuration found by Optuna;
10. run controlled manual comparisons of kernel size, weight decay, patience,
    and learning-rate scheduler.

The manual comparisons keep a reference configuration fixed and change only
one parameter at a time.

The test set remains isolated. It is used only for the final evaluation,
after the configuration and decision threshold have been selected using
development and its validation folds.

The notebook reloads the data and saved splits so that it is self-contained
and reproducible. It does not modify the original signals or saved splits.

This notebook mirrors the workflow of notebook 6, but it uses the non-normalized signal file throughout. The labels, saved patient-level splits, and experimental procedure remain otherwise comparable.

# 6.1. Minimal Setup and Reproducibility

This section loads the non-normalized signals, labels, patient identifiers, and
previously generated data splits.

The following preparation steps are performed:

- convert labels to binary format;
- reshape signals to the format expected by `Conv1d`;
- split development and test using the saved indices;
- retrieve the training and validation indices for all five folds;
- configure the random seeds to promote reproducibility.

No model is trained yet. The purpose is to prepare the structures used by all
subsequent experiments.

In [1]:
# Import the required libraries.
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import copy
import optuna

In [2]:
import sys
print(sys.executable)

/Users/ainhoa.ns/Desktop/TurraFM/.venv/bin/python


In [3]:
# Define the paths to the prepared files.

non_normalized_data_path = (
    Path("data")
    / "nonormalized_signals_clean.npz"
)


dataset_splits_path = (
    Path("data")
    / "splits"
    / "dataset_splits.npz"
)


for file_path in [
    non_normalized_data_path,
    dataset_splits_path,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"File does not exist: {file_path}"
        )

In [4]:
non_normalized_data = np.load(
    non_normalized_data_path
)

required_keys = {
    "signals",
    "label",
    "patient_id",
    "lead",
    "mat_path",
}

missing_keys = required_keys - set(
    non_normalized_data.files
)

if missing_keys:
    raise KeyError(
        f"Missing keys in the .npz file: "
        f"{sorted(missing_keys)}"
    )

print(
    "Available keys:",
    non_normalized_data.files,
)

# Extract the variables 
non_normalized_signals_array = non_normalized_data[
        "signals"
    ].copy()

non_normalized_original_labels = non_normalized_data[
    "label"
].astype(np.int64, copy=True)

# Convert the labels to binary format.
non_normalized_binary_labels = np.where(
    non_normalized_original_labels == 1,
    1,
    0,
).astype(np.int64)

non_normalized_patient_ids = non_normalized_data[
    "patient_id"
].astype(str, copy=True)

non_normalized_leads = non_normalized_data[
    "lead"
].astype(str, copy=True)

non_normalized_mat_paths = non_normalized_data[
    "mat_path"
].astype(str, copy=True)


Available keys: ['signals', 'patient_id', 'lead', 'label', 'mat_path']


In [5]:
# Convert the signals to the Conv1d format.
# The new dimension represents one channel.

# Retrieve the global splits.
# Load the split file.
loaded_splits = np.load(
    dataset_splits_path
) 

development_indices = loaded_splits[
    "development_indices"
]

test_indices = loaded_splits[
    "test_indices"
]


non_normalized_signals_tensor = torch.as_tensor(
    non_normalized_signals_array,
    dtype=torch.float32,
).unsqueeze(
    1
)

# Convert the labels to float32 tensors.
non_normalized_labels_tensor = torch.as_tensor(
    non_normalized_binary_labels,
    dtype=torch.float32,
)


# Retrieve the indices for the five folds.
saved_fold_indices = {}

for fold_number in range(1, 6):
    saved_fold_indices[fold_number] = {
        "train_indices": loaded_splits[
            f"fold_{fold_number}_train_indices"
        ].astype(np.int64, copy=True),

        "val_indices": loaded_splits[
            f"fold_{fold_number}_val_indices"
        ].astype(np.int64, copy=True),
    }

# Create the development and test subsets.
non_normalized_signals_dev_tensor = non_normalized_signals_tensor[development_indices]
non_normalized_labels_dev_tensor = non_normalized_labels_tensor[development_indices]

non_normalized_signals_test_tensor = non_normalized_signals_tensor[test_indices]
non_normalized_labels_test_tensor = non_normalized_labels_tensor[test_indices]

print("Shape of all signals:", non_normalized_signals_tensor.shape)
print("Shape of development signals:", non_normalized_signals_dev_tensor.shape)
print("Shape of test signals:", non_normalized_signals_test_tensor.shape)

Shape of all signals: torch.Size([10577, 1, 252])
Shape of development signals: torch.Size([9359, 1, 252])
Shape of test signals: torch.Size([1218, 1, 252])


In [6]:
# Define a global seed to promote reproducibility.
random_seed = 43

# Set the Python.
random.seed(random_seed)

# Set the NumPy.
np.random.seed(random_seed)

# Set the PyTorch.
torch.manual_seed(random_seed)

# If CUDA is available, set its seeds as well.
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(
        random_seed
    )

# Estas opciones afectan principalmente a CUDA/cuDNN.
# Configure these options to promote reproducible results.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(
    "Seed configured:",
    random_seed,
)

if torch.cuda.is_available():
    print("CUDA available.")
else:
    print("CUDA unavailable. CPU will be used.")

# Store the device explicitly so all later cells use the same setting.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device selected:", device)

Seed configured: 43
CUDA unavailable. CPU will be used.
Device selected: cpu


### Dataset for PyTorch

The `Dataset` class connects the prepared tensors to the PyTorch
DataLoaders.

Each item returns one signal and its label. This class is reused in the
initial checks and when training the five folds.

In [7]:
class ECGDataset(Dataset):
    """
    Dataset of ECG signals prepared for PyTorch.
    """

    def __init__( # Runs when the Dataset is created. Stores signals in self.signals and labels in self.labels.
        self,
        signals,
        labels,
    ): # Store signals and labels as float32 tensors, the usual PyTorch data type.
        # Signals with shape (N, 1, 252).
        self.signals = signals

        # Binary labels with shape (N,).
        self.labels = labels

    def __len__(self):
        # Return the total number of signals.
        return len(self.signals)

    def __getitem__(self, index):
        # Retrieve one signal and its label.
        signal = self.signals[index]
        label = self.labels[index]

        return signal, label

# 6.2. DataLoader Setup and Verification

This section creates the Fold 1 DataLoaders to verify that the input pipeline
works before any model is trained.

The initial configuration uses `batch_size=32`. Later, Optuna explores other
batch sizes within its search space.

The training DataLoader uses `shuffle=True` so that the sample order changes
between epochs. The validation and test DataLoaders use `shuffle=False`
because they do not update model parameters and should preserve a stable
order.

The `torch.Generator` controls the training order and allows us to verify
that equivalent DataLoaders produce the same first batch when given the same
seed.

In [8]:
# Independent generator to control training shuffling.
train_generator = torch.Generator()

train_generator.manual_seed(
    random_seed
)

In [9]:
def create_fold_dataloaders(
    train_indices,
    val_indices,
    batch_size,
    generator=None,
):
    """
    Create the Dataset and DataLoader for one fold.

    The received indices are relative to development.
    """

    train_indices_tensor = torch.as_tensor(
        train_indices,
        dtype=torch.long,
    )

    val_indices_tensor = torch.as_tensor(
        val_indices,
        dtype=torch.long,
    )

    train_signals = non_normalized_signals_dev_tensor[train_indices_tensor]
    train_labels = non_normalized_labels_dev_tensor[train_indices_tensor]

    val_signals = non_normalized_signals_dev_tensor[val_indices_tensor]
    val_labels = non_normalized_labels_dev_tensor[val_indices_tensor]

    train_dataset = ECGDataset(
        train_signals,
        train_labels,
    )

    val_dataset = ECGDataset(
        val_signals,
        val_labels,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return (
        train_dataset,
        val_dataset,
        train_loader,
        val_loader,
    )

### 6.2.1. Quick Checks with Fold 1

These cells only verify the construction of the DataLoaders and the format of
a Fold 1 batch.

They do not run a complete training epoch. The checks are intended to detect
shape, data-type, index, or size errors before the computationally expensive
experiments begin.

In [10]:
# Batch size for the experimental pipeline. Run checks for 32, 64, and 128
batch_size = 32

# Retrieve the current Fold 1 indices.
fold1_train_indices = (
    saved_fold_indices[
        1
    ]["train_indices"]
)

fold1_val_indices = (
    saved_fold_indices[
        1
    ]["val_indices"]
)

# Create the Fold 1 DataLoaders.
(
    fold1_train_dataset,
    fold1_val_dataset,
    fold1_train_loader,
    fold1_val_loader,
) = create_fold_dataloaders(
    train_indices=fold1_train_indices,
    val_indices=fold1_val_indices,
    batch_size=batch_size,
    generator=train_generator,
)

# Create the test Dataset and DataLoader.
test_dataset = ECGDataset(
    non_normalized_signals_test_tensor,
    non_normalized_labels_test_tensor,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

# Minimal checks.
assert len(fold1_train_dataset) > 0
assert len(fold1_val_dataset) > 0
assert len(test_dataset) > 0

# Check the expected sizes.
print("Set sizes:")
print(
    "Train Fold 1:",
    len(fold1_train_dataset),
)
print(
    "Validation Fold 1:",
    len(fold1_val_dataset),
)
print(
    "Test:",
    len(test_dataset),
)


print("\nDataLoaders created successfully.")

Set sizes:
Train Fold 1: 7528
Validation Fold 1: 1831
Test: 1218

DataLoaders created successfully.


In [11]:
# Extract a training batch from Fold 1.
signals_batch, labels_batch = next(
    iter(fold1_train_loader)
)

# Check the shapes and data types.
print(
    "Signal shape:",
    signals_batch.shape,
)

print(
    "Label shape:",
    labels_batch.shape,
)

print(
    "Signal dtype:",
    signals_batch.dtype,
)

print(
    "Label dtype:",
    labels_batch.dtype,
)

assert signals_batch.shape == (
    batch_size,
    1,
    252,
)

assert labels_batch.shape == (batch_size,)

assert signals_batch.dtype == torch.float32
assert labels_batch.dtype == torch.float32

Signal shape: torch.Size([32, 1, 252])
Label shape: torch.Size([32])
Signal dtype: torch.float32
Label dtype: torch.float32


In [12]:
# This block checks that the DataLoader produces the same random order when using the same seed.
# Create two different generators with the same seed.

generator_a = torch.Generator()
generator_b = torch.Generator()

generator_a.manual_seed(
    random_seed
)

generator_b.manual_seed(
    random_seed
)

# Create two equivalent Fold 1 DataLoaders.
(
    train_dataset_a,
    val_dataset_a,
    train_loader_a,
    val_loader_a,
) = create_fold_dataloaders(
    train_indices=fold1_train_indices,
    val_indices=fold1_val_indices,
    batch_size=batch_size,
    generator=generator_a,
)

(
    train_dataset_b,
    val_dataset_b,
    train_loader_b,
    val_loader_b,
) = create_fold_dataloaders(
    train_indices=fold1_train_indices,
    val_indices=fold1_val_indices,
    batch_size=batch_size,
    generator=generator_b,
)

# Extract the first batch from both DataLoaders.
batch_a = next(
    iter(train_loader_a)
)

batch_b = next(
    iter(train_loader_b)
)

signals_equal = torch.equal(
    batch_a[0],
    batch_b[0],
)

labels_equal = torch.equal(
    batch_a[1],
    batch_b[1],
)

print(
    "First-batch signals identical:",
    signals_equal,
)

print(
    "First-batch labels identical:",
    labels_equal,
)

assert signals_equal
assert labels_equal

print(
    "DataLoader reproducibility "
    "verified successfully."
)

First-batch signals identical: True
First-batch labels identical: True
DataLoader reproducibility verified successfully.


# 6.3. Class Imbalance Handling

The labels are imbalanced, so a separate `pos_weight` is calculated for each
fold.

To prevent information leakage, each weight is calculated exclusively from
the labels in that fold's training subset:

`pos_weight = number of negative samples / number of positive samples`

Validation and test do not contribute to this calculation. The weight is
used by `BCEWithLogitsLoss` during training, while the later classification
metrics are calculated without class weighting.

In [13]:
# Dictionary storing the pos_weight for each fold.
fold_pos_weights = {}

# Auxiliary dictionary containing the counts for each fold.
fold_class_counts = {}

# Binary labels for the development set.
non_normalized_y_dev = non_normalized_binary_labels[development_indices]
non_normalized_y_test = non_normalized_binary_labels[test_indices]

# Calculate an independent weight for each fold.
for fold_number in range(1, 6):
    
    # Retrieve only the training indices.
    train_indices = (
        saved_fold_indices[
            fold_number
        ]["train_indices"]
    )
    
    # Select only the training labels.
    train_labels = non_normalized_y_dev[
        train_indices
    ]
    
    # Count positive and negative samples.
    positive_count = np.sum(
        train_labels == 1
    )
    
    negative_count = np.sum(
        train_labels == 0
    )
    
    # Check that the counts cover the entire training set.
    assert (
        positive_count
        + negative_count
        == len(train_indices)
    )
    
    # Calculate the positive-class weight.
    pos_weight_value = (
        negative_count
        / positive_count
    )
    
    # Store the weight as torch.float32.
    fold_pos_weights[
        fold_number
    ] = torch.tensor(
        pos_weight_value,
        dtype=torch.float32,
    )
    
    # Store the counts as well.
    fold_class_counts[
        fold_number
    ] = {
        "positive": int(positive_count),
        "negative": int(negative_count),
    }
    
    # Check that the weight is valid.
    assert np.isfinite(pos_weight_value)
    assert pos_weight_value > 0
    
    # Percentage of positives in the training set.
    positive_percentage = (
        100
        * positive_count
        / len(train_indices)
    )
    
    print(f"\nFold {fold_number}")
    print(
        "  Positive samples:",
        positive_count,
    )
    print(
        "  Negative samples:",
        negative_count,
    )
    print(
        "  Positive percentage:",
        f"{positive_percentage:.2f}%",
    )
    print(
        "  pos_weight:",
        f"{pos_weight_value:.6f}",
    )


for fold_number in range(1, 6):
    print(
        f"Fold {fold_number}:",
        fold_pos_weights[
            fold_number
        ].item(),
    )        


Fold 1
  Positive samples: 1376
  Negative samples: 6152
  Positive percentage: 18.28%
  pos_weight: 4.470930

Fold 2
  Positive samples: 1240
  Negative samples: 6167
  Positive percentage: 16.74%
  pos_weight: 4.973387

Fold 3
  Positive samples: 1418
  Negative samples: 5985
  Positive percentage: 19.15%
  pos_weight: 4.220733

Fold 4
  Positive samples: 1494
  Negative samples: 5979
  Positive percentage: 19.99%
  pos_weight: 4.002008

Fold 5
  Positive samples: 1432
  Negative samples: 6193
  Positive percentage: 18.78%
  pos_weight: 4.324721
Fold 1: 4.470930099487305
Fold 2: 4.973387241363525
Fold 3: 4.220733642578125
Fold 4: 4.002007961273193
Fold 5: 4.324720859527588


# 6.4. Final Experimental Pipeline Verification

The main checks completed so far are grouped here:

- signal and label shapes and data types;
- separation between development and test;
- no patients shared between training and validation;
- integrity of all five folds;
- availability of the DataLoaders;
- availability of the fold-specific `pos_weight` values;
- reproducibility of batch ordering;
- finite signal values.

If these checks complete successfully, the data pipeline is ready for the 1D
CNN definition and training.

In [14]:
# Check that development and test have no patients in common.
development_patients = np.unique(
    non_normalized_patient_ids[
        development_indices
    ]
)

test_patients = np.unique(
    non_normalized_patient_ids[
        test_indices
    ]
)

assert len(
    np.intersect1d(
        development_patients,
        test_patients,
    )
) == 0

# Check the five folds.
for fold_number in range(1, 6):
    
    train_indices = (
        saved_fold_indices[
            fold_number
        ]["train_indices"]
    )
    
    val_indices = (
        saved_fold_indices[
            fold_number
        ]["val_indices"]
    )
    
    
    # There must be no shared indices.
    assert len(
        np.intersect1d(
            train_indices,
            val_indices,
        )
    ) == 0
    
    # Retrieve the training and validation patients.
    train_patients = np.unique(
        non_normalized_patient_ids[
            development_indices[
                train_indices
            ]
        ]
    )
    
    val_patients = np.unique(
        non_normalized_patient_ids[
            development_indices[
                val_indices
            ]
        ]
    )
    
    # Training and validation must not share patients.
    assert len(
        np.intersect1d(
            train_patients,
            val_patients,
        )
    ) == 0


# Check that the DataLoaders contain data.
assert len(fold1_train_loader) > 0
assert len(fold1_val_loader) > 0
assert len(test_loader) > 0

# Check a training batch.
final_batch = next(
    iter(fold1_train_loader)
)

assert final_batch[0].shape == (
    batch_size,
    1,
    252,
)

assert final_batch[1].shape == (batch_size,)

# Check the pos_weight values.
assert len(fold_pos_weights) == 5

for fold_number in range(1, 6):
    assert (
        fold_pos_weights[
            fold_number
        ].dtype
        == torch.float32
    )

# Check reproducibility.
assert signals_equal
assert labels_equal

print("Complete dataset: correct.")
print("Binary labels: correct.")
print("Development and test: correct.")
print("Patient-level folds: correct.")
print("DataLoaders: correct.")
print("pos_weight values: correct.")
print("Reproducibility: correct.")

print(
    "\nExperimental pipeline prepared successfully "
    "to define and train the 1D CNN."
)

Complete dataset: correct.
Binary labels: correct.
Development and test: correct.
Patient-level folds: correct.
DataLoaders: correct.
pos_weight values: correct.
Reproducibility: correct.

Experimental pipeline prepared successfully to define and train the 1D CNN.


# 6.5. Definition and Verification of the Baseline 1D CNN

The initial CNN architecture is defined as the baseline for the experiments
before the Optuna search.

The network contains three convolutional blocks with the following channel
progression:

`1 → 16 → 32 → 64`

Each block contains `Conv1d`, `BatchNorm1d`, `ReLU`, and `MaxPool1d`.
After the convolutional blocks, the model applies:

1. `AdaptiveAvgPool1d(1)` to summarize the temporal dimension;
2. `Dropout(0.3)` for regularization;
3. `Linear(64, 1)` to produce one logit per signal.

The baseline uses kernel size 7 and padding 3, preserving the length before
each pooling operation. This section only verifies model construction; the
complete baseline training is performed later, before Optuna.

In [15]:
class ECGCNN1D(nn.Module):
    def __init__(
        self,
        channels=(16, 32, 64),
        dropout=0.0,
        kernel_size=7,
    ):
        super().__init__()

        if kernel_size <= 0 or kernel_size % 2 == 0:
            raise ValueError(
                "kernel_size must be a positive odd integer."
            )

        layers = []
        in_channels = 1
        padding = kernel_size // 2

        for out_channels in channels:
            layers.extend([
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                ),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2),
            ])
            in_channels = out_channels

        self.features = nn.Sequential(*layers)

        # Reduce the temporal dimension before the classifier.
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            in_channels,
            1,
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        return self.classifier(x).squeeze(1)

### Architecture Instantiation and Summary

The baseline CNN is instantiated and its complete structure is displayed.

The total and trainable parameter counts are also reported. This documents
the model size and verifies that the architecture was built as intended.

In [16]:
# Instantiate the baseline model.
# The Linear layer already has a defined size, so it is not necessary
# to run an artificial forward pass to initialize it.
model = ECGCNN1D()

# Display the complete architecture.
print(model)

# Calculate the total number of parameters.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

# Calculate the number of trainable parameters.
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "\nTotal number of parameters:",
    total_parameters,
)

print(
    "Number of trainable parameters:",
    trainable_parameters,
)

ECGCNN1D(
  (features): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(16, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (global_pool): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.0, inplace=False)
  (classifier): Linear(in_features=64, out_features=1, bias=True)
)

Total number of paramet

### Test Forward Pass with a Real Batch

A real Fold 1 batch is passed through the untrained baseline architecture.

The check verifies that:

- the input has batch, channel, and temporal-length dimensions;
- the network returns one logit per signal;
- the output has the expected shape;
- all output values are finite.

These logits are not performance measurements because the model has not been
trained yet.

In [17]:
# Extract a real batch from the Fold 1 DataLoader.
input_batch, batch_labels = next(
    iter(fold1_train_loader)
)

print(
    "Input batch shape:",
    input_batch.shape,
)

print(
    "Batch dtype:",
    input_batch.dtype,
)

# Put the model in evaluation mode to perform only
# a forward-pass check.
model.eval()

# Disable gradient computation because training has not started yet.
with torch.no_grad():
    output_batch = model(
        input_batch
    )

print(
    "Output shape:",
    output_batch.shape,
)

# Check the input shape.
assert input_batch.ndim == 3
assert input_batch.shape[1] == 1
assert input_batch.shape[2] == 252

# Check the output shape.
assert output_batch.ndim == 1
assert (
    output_batch.shape[0]
    == input_batch.shape[0]
)

# Check that the logits are finite.
assert torch.isfinite(
    output_batch
).all()

print(
    "\nThe forward pass completed successfully."
)

# Display the first five logits.
# These values are random because the model has not been trained yet.
print(
    "Primeros cinco logits:",
    output_batch[:5],
)

Input batch shape: torch.Size([32, 1, 252])
Batch dtype: torch.float32
Output shape: torch.Size([32])

The forward pass completed successfully.
Primeros cinco logits: tensor([0.8632, 0.2873, 0.3931, 0.2039, 0.3118])


### Baseline Architecture Check Result

This result only confirms that the CNN accepts the pipeline inputs and
produces an output compatible with binary classification.

It does not represent predictive performance yet. Metrics require training
the architecture on all five folds and evaluating its validation predictions.

# 6.6. Definition and Verification of the Loss Function

`BCEWithLogitsLoss` is used because the model returns logits and does not
apply sigmoid internally. The loss combines sigmoid and binary cross-entropy
in a numerically stable way.

Each fold uses its own `pos_weight`, calculated in Section 6.3.

The next cell computes a loss check with a Fold 1 batch. It does not run
backpropagation, update parameters, or constitute a training epoch.

In [18]:
def create_loss_function(fold_number):
    """
    Create the BCEWithLogitsLoss function for one fold.

    The pos_weight is retrieved from the values calculated
    previously in fold_pos_weights.
    """

    # Check that the weight for the fold exists.
    if fold_number not in fold_pos_weights:
        raise KeyError(
            f"No pos_weight exists for Fold {fold_number}."
        )

    # Retrieve the previously calculated weight.
    current_pos_weight = fold_pos_weights[
        fold_number
    ]

    # Check that it has the expected type.
    assert isinstance(
        current_pos_weight,
        torch.Tensor,
    )

    assert (
        current_pos_weight.dtype
        == torch.float32
    )

    # Create BCEWithLogitsLoss without applying sigmoid manually.
    loss_function = (
        torch.nn.BCEWithLogitsLoss(
            pos_weight=current_pos_weight,
        )
    )

    return loss_function

### Loss Function Check on Fold 1

Fold 1 is used only as an example to verify that the loss receives compatible
logit and label shapes and uses the correct `pos_weight`.

This check cannot be used to compare models or measure performance.

In [19]:
# Create the loss function for Fold 1.
fold1_loss_function = (
    create_loss_function(1)
)

# Retrieve the pos_weight used.
fold1_pos_weight = fold_pos_weights[1]

print(
    "Loss function:",
    fold1_loss_function,
)

print(
    "pos_weight del Fold 1:",
    fold1_loss_function.pos_weight.item(),
)

# Check that exactly the correct weight was used
# calculated previously for Fold 1.
assert torch.equal(
    fold1_loss_function.pos_weight,
    fold1_pos_weight,
)

print(
    "\nThe Fold 1 pos_weight matches the previously calculated value."
)

Loss function: BCEWithLogitsLoss()
pos_weight del Fold 1: 4.470930099487305

The Fold 1 pos_weight matches the previously calculated value.


In [20]:
# Extract a real batch from the Fold 1 DataLoader.
loss_signals_batch, loss_labels_batch = next(
    iter(fold1_train_loader)
)

# The model is used in evaluation mode only to perform
# a forward-pass and loss-function check.
model.eval()

# Gradients are not computed because training has not started yet.
with torch.no_grad():
    
    # The model returns logits without applying sigmoid.
    loss_logits = model(
        loss_signals_batch
    )
    
    # BCEWithLogitsLoss applies the required transformation internally.
    test_loss = fold1_loss_function(
        loss_logits,
        loss_labels_batch,
    )

# Display the check results.
print(
    "Logit shape:",
    loss_logits.shape,
)

print(
    "Label shape:",
    loss_labels_batch.shape,
)

print(
    "pos_weight used:",
    fold1_loss_function.pos_weight.item(),
)

print(
    "Test loss value:",
    test_loss.item(),
)

# Check that logits and labels have the same shape.
assert (
    loss_logits.shape
    == loss_labels_batch.shape
)

# Check the data types.
assert loss_logits.dtype == torch.float32
assert loss_labels_batch.dtype == torch.float32

# A scalar loss has zero dimensions.
assert test_loss.ndim == 0

# Check that the loss is finite.
assert torch.isfinite(test_loss)

# Check that the Fold 1 pos_weight is preserved.
assert torch.equal(
    fold1_loss_function.pos_weight,
    fold_pos_weights[1],
)

print(
    "\nBCEWithLogitsLoss works correctly."
)

Logit shape: torch.Size([32])
Label shape: torch.Size([32])
pos_weight used: 4.470930099487305
Test loss value: 1.2856624126434326

BCEWithLogitsLoss works correctly.


In [21]:
# Create the loss functions for the five folds.
# All of them reuse the previously calculated pos_weight values.
fold_loss_functions = {}

for fold_number in range(1, 6):
    
    fold_loss_functions[
        fold_number
    ] = create_loss_function(
        fold_number
    )

# Check that one function was created for each fold.
assert len(fold_loss_functions) == 5

for fold_number in range(1, 6):
    
    current_loss_function = (
        fold_loss_functions[
            fold_number
        ]
    )
    
    expected_pos_weight = fold_pos_weights[
        fold_number
    ]
    
    # Check that each function uses its corresponding weight.
    assert torch.equal(
        current_loss_function.pos_weight,
        expected_pos_weight,
    )

print(
    "BCEWithLogitsLoss functions created successfully for all five folds."
)

BCEWithLogitsLoss functions created successfully for all five folds.


### Loss Function Check Result

This verifies that `BCEWithLogitsLoss` works with the network output, that
the `pos_weight` corresponds to Fold 1, and that the loss is a finite scalar.

The magnitude of this isolated loss must not be interpreted as model
performance because complete training has not taken place yet.

# 6.7. Definition and Verification of the Optimizer

Adam is used as the initial optimizer.

The initial architecture uses `learning_rate=3e-4`. This value is the
reference configuration for baseline training; Optuna later explores a range
of learning rates.

This section verifies that Adam is connected to the trainable parameters. No
parameter update is performed yet.

In [22]:
# Initial optimizer learning rate.
# This value is not necessarily optimal.
learning_rate = 3e-4

print(
    "Learning rate initial:",
    learning_rate,
)

def create_optimizer(
    model,
    learning_rate,
    weight_decay=0.0,
):
    """
    Create an Adam optimizer for the model parameters.
    """

    # Adam is connected to all model parameters.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    return optimizer


optimizer = create_optimizer(
    model= model,  # replace model with the actual model variable name
    learning_rate = learning_rate,
)

Learning rate initial: 0.0003


In [23]:
# Check that the optimizer is Adam.
assert isinstance(
    optimizer,
    torch.optim.Adam,
)

# There must be at least one parameter group.
assert len(optimizer.param_groups) >= 1

# Check that all groups use the defined learning rate.
for param_group in optimizer.param_groups:
    assert (
        param_group["lr"]
        == learning_rate
    )

# Retrieve the trainable model parameters.
model_trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

# Retrieve the parameters connected to the optimizer.
optimizer_parameters = []

for param_group in optimizer.param_groups:
    optimizer_parameters.extend(
        param_group["params"]
    )

# Check that the optimizer contains model parameters.
assert len(optimizer_parameters) > 0

# Check that all trainable model parameters
# are included in the optimizer.
assert {
    id(parameter)
    for parameter in model_trainable_parameters
} == {
    id(parameter)
    for parameter in optimizer_parameters
}

print(
    "Adam is correctly connected "
    "to the trainable model parameters."
)
print(
    "Optimizer ready for subsequent training."
)

Adam is correctly connected to the trainable model parameters.
Optimizer ready for subsequent training.


### Optimizer Check Result

This cell confirms that Adam receives the model's trainable parameters.

No training epoch has been executed yet. Parameter updates begin only in the
baseline-training, Optuna, and manual-comparison sections.

# 6.8. Training and Early Stopping Configuration

The common training-loop settings are fixed here:

- maximum number of epochs;
- early-stopping patience;
- `min_delta`;
- validation-loss monitoring;
- preservation of the best model state.

Validation loss is used to select and restore the best state. Accuracy is not
used as the stopping criterion because the classes are imbalanced. Metrics
such as PR-AUC, recall, F1, and balanced accuracy are calculated later from
validation predictions.

In [24]:
# Maximum number of allowed epochs.
# This is an initial configuration, not necessarily an optimal one.
max_epochs = 100

# Maximum number of consecutive epochs without improvement
# before stopping training.
patience = 10

# Minimum improvement required in validation loss.
# With min_delta=0, any decrease counts as an improvement.
min_delta = 0.0

# Lists prepared to record losses during
# subsequent training.
train_loss_history = []
val_loss_history = []

print(
    "Maximum number of epochs:",
    max_epochs,
)

print(
    "Patience:",
    patience,
)

print(
    "Min delta:",
    min_delta,
)

print(
    "Monitored criterion: validation loss"
)

Maximum number of epochs: 100
Patience: 10
Min delta: 0.0
Monitored criterion: validation loss


### EarlyStopping Class

This class implements the logic that controls when training stops.

When validation loss improves, the best value is updated. When it does not
improve, the counter of epochs without improvement increases. Training stops
when the patience value is reached.

The class does not train or save models by itself; it only provides the state
used by the training loop.

In [25]:
class EarlyStopping:
    """
    Control sencillo del early stopping mediante validation loss.
    """

    def __init__(
        self,
        patience,
        min_delta,
    ):
        # Maximum number of epochs without improvement.
        self.patience = patience

        # Minimum required improvement.
        self.min_delta = min_delta

        # Best validation loss observed so far.
        self.best_val_loss = float("inf")

        # Number of consecutive epochs without improvement.
        self.epochs_without_improvement = 0

        # Indicates whether training should stop.
        self.stop_training = False

    def step(self, current_val_loss):
        """
        Update the state using the current validation loss.
        """

        # An improvement occurs when the loss decreases
        # by at least min_delta relative to the previous best value.
        if current_val_loss < (
            self.best_val_loss - self.min_delta
        ):
            self.best_val_loss = current_val_loss
            self.epochs_without_improvement = 0

        else:
            # No improvement occurred.
            self.epochs_without_improvement += 1

            # Activamos early stopping al alcanzar patience.
            if (
                self.epochs_without_improvement
                >= self.patience
            ):
                self.stop_training = True

        return self.stop_training

### Artificial Early Stopping Test

A fictitious validation-loss sequence is used to test the patience logic
without processing ECG signals.

The test verifies that the counter increases when there is no improvement and
that stopping is triggered when patience is reached. This is a logic check,
not an experimental model result.

In [26]:
# Create an artificial validation-loss sequence.
# The loss initially improves and then remains constant.
artificial_val_losses = [
    1.00,
    0.80,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
    0.70,
]

# Create an independent instance for the test.
early_stopping_test = EarlyStopping(
    patience=patience,
    min_delta=min_delta,
)

# Process the artificial sequence.
for epoch_number, current_val_loss in enumerate(
    artificial_val_losses,
    start=1,
):
    stop_training = (
        early_stopping_test.step(
            current_val_loss
        )
    )

    print(
        f"epoch {epoch_number}: "
        f"val_loss={current_val_loss:.2f}, "
        f"best_val_loss="
        f"{early_stopping_test.best_val_loss:.2f}, "
        f"sin_mejora="
        f"{early_stopping_test.epochs_without_improvement}, "
        f"stop_training={stop_training}"
    )

    # When early stopping is triggered, end the test.
    if stop_training:
        break

epoch 1: val_loss=1.00, best_val_loss=1.00, sin_mejora=0, stop_training=False
epoch 2: val_loss=0.80, best_val_loss=0.80, sin_mejora=0, stop_training=False
epoch 3: val_loss=0.70, best_val_loss=0.70, sin_mejora=0, stop_training=False
epoch 4: val_loss=0.70, best_val_loss=0.70, sin_mejora=1, stop_training=False
epoch 5: val_loss=0.70, best_val_loss=0.70, sin_mejora=2, stop_training=False
epoch 6: val_loss=0.70, best_val_loss=0.70, sin_mejora=3, stop_training=False
epoch 7: val_loss=0.70, best_val_loss=0.70, sin_mejora=4, stop_training=False
epoch 8: val_loss=0.70, best_val_loss=0.70, sin_mejora=5, stop_training=False
epoch 9: val_loss=0.70, best_val_loss=0.70, sin_mejora=6, stop_training=False
epoch 10: val_loss=0.70, best_val_loss=0.70, sin_mejora=7, stop_training=False
epoch 11: val_loss=0.70, best_val_loss=0.70, sin_mejora=8, stop_training=False
epoch 12: val_loss=0.70, best_val_loss=0.70, sin_mejora=9, stop_training=False
epoch 13: val_loss=0.70, best_val_loss=0.70, sin_mejora=10, s

In [27]:
# Check that the best observed value is 0.70.
assert (
    early_stopping_test.best_val_loss
    == 0.70
)

# Check that the counter has reached patience.
assert (
    early_stopping_test.epochs_without_improvement
    >= patience
)

# Check that stopping was triggered.
assert (
    early_stopping_test.stop_training
    is True
)

print(
    "\nThe artificial early-stopping test "
    "completed successfully."
)

print(
    "Best validation loss:",
    early_stopping_test.best_val_loss,
)

print(
    "Epochs without improvement:",
    early_stopping_test.epochs_without_improvement,
)

print(
    "Stop training:",
    early_stopping_test.stop_training,
)

print(
    "\nTraining configuration and "
    "early stopping prepared successfully."
)


The artificial early-stopping test completed successfully.
Best validation loss: 0.7
Epochs without improvement: 10
Stop training: True

Training configuration and early stopping prepared successfully.


# 6.9. Training and Validation Functions for One Epoch

Reusable functions are defined for one training epoch and one validation
pass.

The training function computes logits and loss, backpropagates gradients, and
updates the parameters. The validation function disables gradients and does
not modify the model.

Mean losses are weighted by the actual number of samples in each batch, so a
smaller final batch does not receive the same weight as a full batch.

In [28]:
def train_one_epoch(
    model,
    dataloader,
    loss_function,
    optimizer,
    device,
):
    """
    Train the model for one epoch.
    """

    # Enable training mode.
    model.train()

    # Move the loss function to the selected device.
    loss_function = loss_function.to(device)

    # Accumulators for the weighted mean loss.
    total_loss = 0.0
    total_samples = 0

    # Iterate over all training batches.
    for signals, labels in dataloader:
        
        # Move signals and labels to the device.
        signals = signals.to(device)
        labels = labels.to(device)

        # Clear gradients accumulated from the previous batch.
        optimizer.zero_grad()

        # Compute the model logits.
        logits = model(signals)

        # Check that logits and labels have the same shape.
        assert logits.shape == labels.shape

        # Compute the batch loss.
        loss = loss_function(logits, labels)

        # Compute the gradients.
        loss.backward()

        # Update the model parameters.
        optimizer.step()

        # current number of signals in the current batch.
        batch_size_actual = signals.size(0)

        # Accumulate the loss weighted by batch size.
        total_loss += loss.item() * batch_size_actual
        total_samples += batch_size_actual

    # Compute the mean loss over all samples.
    mean_loss = total_loss / total_samples

    return mean_loss

In [29]:
def validate_one_epoch(
    model,
    dataloader,
    loss_function,
    device,
):
    """
    Evaluate the model during one validation pass.
    """

    # Enable evaluation mode.
    model.eval()

    # Move the loss function to the selected device.
    loss_function = loss_function.to(device)

    # Accumulators for the mean validation loss.
    total_loss = 0.0
    total_samples = 0

    # Disable gradient computation.
    with torch.no_grad():
        
        # Iterate over all validation batches.
        for signals, labels in dataloader:
            
            # Move signals and labels to the device.
            signals = signals.to(device)
            labels = labels.to(device)

            # Compute the logits.
            logits = model(signals)

            # Check that logits and labels have the same shape.
            assert logits.shape == labels.shape

            # Compute the batch loss.
            loss = loss_function(logits, labels)

            # current number of signals in the current batch.
            batch_size_actual = signals.size(0)

            # Accumulate the loss weighted by batch size.
            total_loss += loss.item() * batch_size_actual
            total_samples += batch_size_actual

    # Compute the mean validation loss.
    mean_loss = total_loss / total_samples

    return mean_loss

In [30]:
print(
    "The baseline architecture has already been evaluated."
)
print(
    "The Optuna search will now be defined and executed."
)
print(
    "The kernel-comparison configuration is fixed independently in Section 6.15."
)


The baseline architecture has already been evaluated.
The Optuna search will now be defined and executed.
The kernel-comparison configuration is fixed independently in Section 6.15.


# 6.10. Setup and Training of the Baseline Architecture

This section completes the training engine and runs the baseline architecture.

The experimental order is deliberate:

1. define the reusable cross-validation function;
2. train the baseline architecture on all five folds;
3. obtain its validation probabilities;
4. analyze its metrics at different thresholds;
5. run Optuna only afterwards.

The baseline establishes the performance obtained before automatic
hyperparameter search.

### Reusable Cross-Validation Engine

The function below trains and validates an independent model on each of the
five folds.

The same engine is first used for the baseline, then reused for each Optuna
trial and for the manual comparisons.

For each configuration, the best state for every fold is preserved according
to validation loss. The test set is not involved in this process.

In [31]:
def run_cross_validation(config):
    """
    Train and validate one architecture on the five folds.

    The received configuration may come from Optuna.
    """

    run_random_seed = config.get("random_seed", random_seed)
    run_max_epochs = config.get("max_epochs", max_epochs)
    run_patience = config.get("patience", patience)
    run_min_delta = config.get("min_delta", min_delta)
    run_device = torch.device(config.get("device", device))

    cross_validation_results = {}
    fold_models = {}

    for fold_number in range(1, 6):

        print(
            f"\n===== Fold {fold_number} ====="
        )

        # Indices relative to the development set.
        train_indices = saved_fold_indices[
            fold_number
        ]["train_indices"]

        val_indices = saved_fold_indices[
            fold_number
        ]["val_indices"]

        # Reproducible and independent generator for each fold.
        fold_generator = torch.Generator()
        fold_generator.manual_seed(
            run_random_seed + fold_number
        )

        # Independent seed for model initialization.
        torch.manual_seed(
            run_random_seed + fold_number
        )

        if torch.cuda.is_available():
            torch.cuda.manual_seed(
                run_random_seed + fold_number
            )
            torch.cuda.manual_seed_all(
                run_random_seed + fold_number
            )

        # Create the model using the current configuration.
        current_model = ECGCNN1D(
            channels=config["channels"],
            dropout=config["dropout"],
            kernel_size=config.get("kernel_size", 7),
        ).to(run_device)

        # Optimizer using the learning rate from the current configuration.
        current_optimizer = create_optimizer(
            model=current_model,
            learning_rate=config["learning_rate"],
            weight_decay=config.get("weight_decay", 0.0),
        )

        current_scheduler = None

        if config.get("use_scheduler", False):
            current_scheduler = (
                torch.optim.lr_scheduler.ReduceLROnPlateau(
                    current_optimizer,
                    mode="min",
                    factor=config.get(
                        "scheduler_factor",
                        0.5,
                    ),
                    patience=config.get(
                        "scheduler_patience",
                        3,
                    ),
                    threshold=config.get(
                        "scheduler_threshold",
                        1e-4,
                    ),
                    threshold_mode="rel",
                    min_lr=config.get(
                        "scheduler_min_lr",
                        1e-6,
                ),
            )
        )

        # Loss function specific to the fold.
        current_loss_function = (
            create_loss_function(fold_number)
        )

        # Independent early stopping for this fold.
        current_early_stopping = EarlyStopping(
            patience=run_patience,
            min_delta=run_min_delta,
        )

        # DataLoaders for the current fold.
        (
            current_train_dataset,
            current_val_dataset,
            current_train_loader,
            current_val_loader,
        ) = create_fold_dataloaders(
            train_indices=train_indices,
            val_indices=val_indices,
            batch_size=config["batch_size"],
            generator=fold_generator,
        )

        # Loss histories.
        train_loss_history = []
        val_loss_history = []
        learning_rate_history = []

        # best estado encontrado.
        best_model_state = None
        best_val_loss = float("inf")
        best_epoch = None

        stopped_by_early_stopping = False

        # Main training loop.
        for epoch_number in range(
            1,
            run_max_epochs + 1,
        ):

            learning_rate_history.append(
                current_optimizer.param_groups[0]["lr"]
            )

            current_train_loss = train_one_epoch(
                model=current_model,
                dataloader=current_train_loader,
                loss_function=current_loss_function,
                optimizer=current_optimizer,
                device=run_device,
            )

            current_val_loss = validate_one_epoch(
                model=current_model,
                dataloader=current_val_loader,
                loss_function=current_loss_function,
                device=run_device,
            )

            # Check that the losses are valid.
            assert np.isfinite(
                current_train_loss
            )

            assert np.isfinite(
                current_val_loss
            )

            if current_scheduler is not None:
                current_scheduler.step(
                    current_val_loss
                )

            train_loss_history.append(
                current_train_loss
            )

            val_loss_history.append(
                current_val_loss
            )

            # Store the best model according to validation loss.
            has_improved = (
                current_val_loss
                < (
                    best_val_loss
                    - run_min_delta
                )
            )

            if has_improved:
                best_val_loss = current_val_loss
                best_epoch = epoch_number

                best_model_state = copy.deepcopy(
                    current_model.state_dict()
                )

            # Actualizamos early stopping.
            should_stop = (
                current_early_stopping.step(
                    current_val_loss
                )
            )

            if should_stop:
                stopped_by_early_stopping = True
                break

        # Training checks.
        epochs_executed = len(
            train_loss_history
        )

        assert epochs_executed > 0
        assert best_model_state is not None
        assert best_epoch is not None

        assert np.isfinite(
            best_val_loss
        )

        assert np.isclose(
            best_val_loss,
            min(val_loss_history),
        )

        # Restore the best model state.
        current_model.load_state_dict(
            best_model_state
        )

        # Store the final model for the fold.
        fold_models[fold_number] = current_model

        if stopped_by_early_stopping:
            stop_reason = "early stopping"
        else:
            stop_reason = "alcanzado max_epochs"

        # Store the results for the fold.
        cross_validation_results[
            fold_number
        ] = {
            "train_loss_history": train_loss_history,
            "val_loss_history": val_loss_history,
            "learning_rate_history": learning_rate_history,
            "scheduler_used": current_scheduler is not None,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "epochs_executed": epochs_executed,
            "stop_reason": stop_reason,
            "best_model_state": copy.deepcopy(
                best_model_state
            ),
            "state_restored_correctly": True,
            "pos_weight": (
                current_loss_function
                .pos_weight
                .item()
            ),
        }

        print(
            f"Fold {fold_number}: "
            f"best epoch={best_epoch}, "
            f"validation loss={best_val_loss:.6f}, "
            f"epochs={epochs_executed}"
        )

    # Final checks.
    assert set(
        cross_validation_results
    ) == set(range(1, 6))

    assert set(
        fold_models
    ) == set(range(1, 6))

    return (
        cross_validation_results,
        fold_models,
    )

## 6.10.1. Training the Baseline Architecture

The proposed baseline is trained before Optuna. This provides a reference
performance without automatic hyperparameter search.

The baseline configuration is:

- channels: `(16, 32, 64)`;
- dropout: `0.0`;
- kernel size: `7`, with automatically calculated padding;
- learning rate: `3e-4`;
- batch size: `32`;
- weight decay: `0`;
- patience: `10`;
- no scheduler.

Training is performed on all five development folds using the same
cross-validation engine later used by Optuna.

In [ ]:
# Train the baseline architecture before Optuna.
#
# This configuration reproduces the initially proposed architecture.
# Train the five folds using development only.

initial_architecture_config = {
    "channels": (16, 32, 64),
    "dropout": 0.0,
    "learning_rate": 3e-4,
    "batch_size": 32,
    "kernel_size": 7,
    "weight_decay": 0.0,
    "random_seed": 43,
    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "device": "cpu",
    "use_scheduler": False,
}

initial_cv_results, initial_fold_models = (
    run_cross_validation(
        initial_architecture_config
    )
)

print(
    "Baseline architecture training "
    "completed for all five folds."
)

In [ ]:
# Evaluate the baseline architecture at several thresholds.
#
# Reconstruct validation probabilities for the five folds.
# The test set is not used.

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)

initial_validation_predictions = {}

for fold_number in range(1, 6):

    val_indices = saved_fold_indices[
        fold_number
    ]["val_indices"]

    val_indices_tensor = torch.as_tensor(
        val_indices,
        dtype=torch.long,
    )

    val_dataset = ECGDataset(
        non_normalized_signals_dev_tensor[val_indices_tensor],
        non_normalized_labels_dev_tensor[val_indices_tensor],
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=initial_architecture_config[
            "batch_size"
        ],
        shuffle=False,
        num_workers=0,
    )

    evaluation_model = ECGCNN1D(
        channels=initial_architecture_config[
            "channels"
        ],
        dropout=initial_architecture_config[
            "dropout"
        ],
        kernel_size=initial_architecture_config[
            "kernel_size"
        ],
    ).to(device)

    with torch.no_grad():
        evaluation_model(
            torch.zeros(
                1,
                1,
                252,
                device=device,
            )
        )

    evaluation_model.load_state_dict(
        initial_cv_results[
            fold_number
        ]["best_model_state"]
    )
    evaluation_model.eval()

    probability_parts = []
    label_parts = []

    with torch.no_grad():
        for signals, labels in val_loader:
            logits = evaluation_model(
                signals.to(device)
            )
            probabilities = torch.sigmoid(
                logits
            )

            probability_parts.append(
                probabilities.cpu().numpy()
            )
            label_parts.append(
                labels.cpu().numpy()
            )

    initial_validation_predictions[
        fold_number
    ] = {
        "y_true": np.concatenate(
            label_parts
        ).astype(np.int64),
        "y_prob": np.concatenate(
            probability_parts
        ).astype(np.float64),
    }


threshold_values = np.round(
    np.arange(0.05, 0.96, 0.05),
    2,
)

global_y_true = np.concatenate([
    initial_validation_predictions[
        fold_number
    ]["y_true"]
    for fold_number in range(1, 6)
])

global_y_prob = np.concatenate([
    initial_validation_predictions[
        fold_number
    ]["y_prob"]
    for fold_number in range(1, 6)
])

global_threshold_records = []

for threshold in threshold_values:

    global_y_pred = (
        global_y_prob >= threshold
    ).astype(np.int64)

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(
            global_y_true,
            global_y_pred,
            labels=[0, 1],
        ).ravel()
    )

    global_threshold_records.append(
        {
            "threshold": threshold,
            "precision": precision_score(
                global_y_true,
                global_y_pred,
                zero_division=0,
            ),
            "recall": recall_score(
                global_y_true,
                global_y_pred,
                zero_division=0,
            ),
            "specificity": true_negative / (
                true_negative + false_positive
            ),
            "f1": f1_score(
                global_y_true,
                global_y_pred,
                zero_division=0,
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    global_y_true,
                    global_y_pred,
                )
            ),
            "false_positives": false_positive,
            "false_negatives": false_negative,
        }
    )

initial_global_threshold_metrics = pd.DataFrame(
    global_threshold_records
)

initial_fold_threshold_records = []

for fold_number in range(1, 6):

    fold_y_true = initial_validation_predictions[
        fold_number
    ]["y_true"]

    fold_y_prob = initial_validation_predictions[
        fold_number
    ]["y_prob"]

    for threshold in threshold_values:

        fold_y_pred = (
            fold_y_prob >= threshold
        ).astype(np.int64)

        true_negative, false_positive, false_negative, true_positive = (
            confusion_matrix(
                fold_y_true,
                fold_y_pred,
                labels=[0, 1],
            ).ravel()
        )

        initial_fold_threshold_records.append(
            {
                "fold_number": fold_number,
                "threshold": threshold,
                "precision": precision_score(
                    fold_y_true,
                    fold_y_pred,
                    zero_division=0,
                ),
                "recall": recall_score(
                    fold_y_true,
                    fold_y_pred,
                    zero_division=0,
                ),
                "specificity": true_negative / (
                    true_negative + false_positive
                ),
                "f1": f1_score(
                    fold_y_true,
                    fold_y_pred,
                    zero_division=0,
                ),
                "balanced_accuracy": (
                    balanced_accuracy_score(
                        fold_y_true,
                        fold_y_pred,
                    )
                ),
                "false_positives": false_positive,
                "false_negatives": false_negative,
            }
        )

initial_fold_threshold_metrics = pd.DataFrame(
    initial_fold_threshold_records
)

print("global results by threshold:")
display(initial_global_threshold_metrics)

print("Results by fold and threshold:")
display(initial_fold_threshold_metrics)

print("global candidates:")

best_global_f1 = initial_global_threshold_metrics.loc[
    initial_global_threshold_metrics["f1"].idxmax()
]

best_global_balanced_accuracy = (
    initial_global_threshold_metrics.loc[
        initial_global_threshold_metrics[
            "balanced_accuracy"
        ].idxmax()
    ]
)

recall_candidates = (
    initial_global_threshold_metrics[
        initial_global_threshold_metrics["recall"] >= 0.90
    ]
)

print(
    "Threshold with maximum F1:",
    best_global_f1["threshold"],
    "| F1:",
    best_global_f1["f1"],
)

print(
    "Threshold with maximum balanced accuracy:",
    best_global_balanced_accuracy["threshold"],
    "| Balanced accuracy:",
    best_global_balanced_accuracy[
        "balanced_accuracy"
    ],
)

if not recall_candidates.empty:
    highest_recall_threshold = recall_candidates.iloc[-1]

    print(
        "Highest threshold with recall >= 0.90:",
        highest_recall_threshold["threshold"],
        "| Recall:",
        highest_recall_threshold["recall"],
    )

### Check with a Manually Selected Threshold

The next cell evaluates the baseline at an explicitly selected decision
threshold. The value can be changed without retraining because validation
probabilities have already been stored.

For each fold, the metrics and confusion matrix are displayed. The matrix is
reported in the order `[[TN, FP], [FN, TP]]`.

The test set is not used in this check.

In [ ]:
# Manually selected threshold for the baseline architecture.
#
# This value can be changed without retraining the models because
# the validation probabilities have already been stored.
initial_selected_threshold = 0.4

assert 0.0 < initial_selected_threshold < 1.0

initial_selected_threshold_records = []

print(
    "Baseline architecture results with "
    f"threshold={initial_selected_threshold:.2f}:"
)

for fold_number in range(1, 6):

    fold_y_true = initial_validation_predictions[
        fold_number
    ]["y_true"]

    fold_y_prob = initial_validation_predictions[
        fold_number
    ]["y_prob"]

    fold_y_pred = (
        fold_y_prob >= initial_selected_threshold
    ).astype(np.int64)

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(
            fold_y_true,
            fold_y_pred,
            labels=[0, 1],
        ).ravel()
    )

    fold_metrics = {
        "fold_number": fold_number,
        "threshold": initial_selected_threshold,
        "precision": precision_score(
            fold_y_true,
            fold_y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            fold_y_true,
            fold_y_pred,
            zero_division=0,
        ),
        "specificity": true_negative / (
            true_negative + false_positive
        ),
        "f1": f1_score(
            fold_y_true,
            fold_y_pred,
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            fold_y_true,
            fold_y_pred,
        ),
        "false_positives": false_positive,
        "false_negatives": false_negative,
    }

    initial_selected_threshold_records.append(
        fold_metrics
    )

    print(f"\nFold {fold_number}")
    print(
        "  Precision:",
        f"{fold_metrics['precision']:.6f}",
    )
    print(
        "  Recall/sensitivity:",
        f"{fold_metrics['recall']:.6f}",
    )
    print(
        "  Specificity:",
        f"{fold_metrics['specificity']:.6f}",
    )
    print(
        "  F1:",
        f"{fold_metrics['f1']:.6f}",
    )
    print(
        "  Balanced accuracy:",
        f"{fold_metrics['balanced_accuracy']:.6f}",
    )
    print(
        "  False positives:",
        false_positive,
    )
    print(
        "  False negatives:",
        false_negative,
    )
    print("  Confusion matrix:")
    print(
        np.array([
            [true_negative, false_positive],
            [false_negative, true_positive],
        ])
    )

initial_selected_threshold_by_fold = pd.DataFrame(
    initial_selected_threshold_records
)

print("\nSummary table by fold:")
display(initial_selected_threshold_by_fold)


# 6.11. Hyperparameter Search with Optuna

The baseline has been trained and its predictions have been analyzed. The
automatic search now evaluates one configuration per trial.

Optuna explores:

- number of convolutional blocks: 2 to 4;
- channels in each block: `16`, `32`, `64`, or `128`;
- dropout: continuous values from `0.0` to `0.5`;
- learning rate: values from `1e-5` to `1e-2` on a logarithmic scale;
- batch size: fixed at `16`;
- weight decay: continuous values from `0.0` to `1e-5`.

The following settings remain fixed during the search:

- kernel size: `5`;
- patience: `10`;
- no scheduler;
- maximum epochs: `100`;
- `min_delta`: `0`;
- random seed: `43`.

Each trial trains the five folds, generates out-of-fold predictions, and
computes a score. For each configuration, a threshold from the grid
`0.05, 0.10, ..., 0.95` is considered valid when global recall is at least
`0.90`. Among valid thresholds, the score maximized is:

`0.5 · balanced accuracy + 0.3 · F1 + 0.2 · precision`.

The search uses development and its validation folds only. The test set
remains reserved for the final evaluation.

In [32]:
# Define the Optuna search space.
# Parameters not included here remain fixed.
def suggest_model_config(trial):
    n_layers = trial.suggest_int(
        "n_layers", 2, 4
    )

    channels = tuple(
        trial.suggest_categorical(
            f"channels_{i}",
            [16, 32, 64, 128]
        )
        for i in range(n_layers)
    )

    return {
        "channels": channels,
        "dropout": trial.suggest_float(
            "dropout", 0.0, 0.5
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-5, 1e-2, log=True
        ),
        # Fixed batch size for this search.
        "batch_size": 16,

        # Weight decay is searched on a linear scale because zero is
        # included in the range.
        "weight_decay": trial.suggest_float(
            "weight_decay", 0.0, 1e-5
        ),

        # Other parameters fixed during the Optuna search.
        "kernel_size": 5,
        "patience": 10,
        "use_scheduler": False,
    }

In [33]:
def objective(trial):
    """Optimize classification metrics, not only validation loss."""
    from sklearn.metrics import (
        balanced_accuracy_score,
        f1_score,
        precision_score,
        recall_score,
    )

    config = suggest_model_config(trial)
    results, models = run_cross_validation(config)
    run_device = torch.device(config.get("device", device))

    # Reconstruct out-of-fold predictions for the development set.
    oof_y_true = np.empty(
        len(development_indices),
        dtype=np.int64,
    )
    oof_y_prob = np.empty(
        len(development_indices),
        dtype=np.float64,
    )

    for fold_number in range(1, 6):
        val_indices = saved_fold_indices[
            fold_number
        ]["val_indices"]

        val_dataset = ECGDataset(
            non_normalized_signals_dev_tensor[
                torch.as_tensor(
                    val_indices,
                    dtype=torch.long,
                )
            ],
            non_normalized_labels_dev_tensor[
                torch.as_tensor(
                    val_indices,
                    dtype=torch.long,
                )
            ],
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=0,
        )

        probabilities = []
        labels = []
        model = models[fold_number]
        model.eval()

        with torch.no_grad():
            for signals, batch_labels in val_loader:
                signals = signals.to(run_device)
                logits = model(signals)
                probabilities.append(
                    torch.sigmoid(logits).cpu().numpy()
                )
                labels.append(
                    batch_labels.numpy()
                )

        oof_y_prob[val_indices] = np.concatenate(
            probabilities
        )
        oof_y_true[val_indices] = np.concatenate(
            labels
        ).astype(np.int64)

    # Find the best threshold while maintaining global recall >= 0.90.
    candidate_thresholds = np.round(
        np.arange(0.05, 0.96, 0.05),
        2,
    )
    target_recall = 0.90
    candidates = []

    for threshold in candidate_thresholds:
        predicted = (
            oof_y_prob >= threshold
        ).astype(np.int64)
        recall = recall_score(
            oof_y_true,
            predicted,
            zero_division=0,
        )
        precision = precision_score(
            oof_y_true,
            predicted,
            zero_division=0,
        )
        f1 = f1_score(
            oof_y_true,
            predicted,
            zero_division=0,
        )
        balanced_accuracy = balanced_accuracy_score(
            oof_y_true,
            predicted,
        )

        if recall >= target_recall:
            score = (
                0.5 * balanced_accuracy
                + 0.3 * f1
                + 0.2 * precision
            )
            candidates.append(
                (score, threshold, recall)
            )

    if candidates:
        score, selected_threshold, selected_recall = max(
            candidates,
            key=lambda item: item[0],
        )
    else:
        # Penalize configurations that do not reach
        # the minimum target recall.
        threshold = 0.25
        predicted = (
            oof_y_prob >= threshold
        ).astype(np.int64)
        selected_recall = recall_score(
            oof_y_true,
            predicted,
            zero_division=0,
        )
        selected_threshold = threshold
        score = (
            -10.0 * (target_recall - selected_recall)
        )

    trial.set_user_attr(
        "selected_threshold",
        float(selected_threshold),
    )
    trial.set_user_attr(
        "oof_recall",
        float(selected_recall),
    )

    return float(score)

In [34]:
# Run the hyperparameter search with Optuna.
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=15,
)

best_params = study.best_params

best_config = {
    "channels": tuple(
        best_params[f"channels_{i}"]
        for i in range(best_params["n_layers"])
    ),
    "dropout": best_params["dropout"],
    "learning_rate": best_params["learning_rate"],
    # Fixed batch size used during the search.
    "batch_size": 16,
    "weight_decay": best_params["weight_decay"],

    # Other parameters fixed during Optuna.
    "kernel_size": 5,
    "patience": 10,
    "use_scheduler": False,
}

selected_threshold = study.best_trial.user_attrs[
    "selected_threshold"
]

print("Best trial:", study.best_trial.number)
print("Best value:", study.best_trial.value)
print("Best parameters:", best_params)
print("Reconstructed configuration:", best_config)
print("Selected threshold:", selected_threshold)

[I 2026-10-29 02:46:55,273] A new study created in memory with name: no-name-455bb538-f00e-4dd1-a13d-270eb035455d



===== Fold 1 =====
Fold 1: best epoch=4, validation loss=1.048632, epochs=14

===== Fold 2 =====
Fold 2: best epoch=8, validation loss=0.905652, epochs=18

===== Fold 3 =====
Fold 3: best epoch=3, validation loss=0.805279, epochs=13

===== Fold 4 =====
Fold 4: best epoch=22, validation loss=0.654645, epochs=32

===== Fold 5 =====
Fold 5: best epoch=24, validation loss=0.364063, epochs=34


[I 2026-10-29 04:26:03,206] Trial 0 finished with value: 0.5750027595657582 and parameters: {'n_layers': 3, 'channels_0': 32, 'channels_1': 64, 'channels_2': 128, 'dropout': 0.33854678128865456, 'learning_rate': 9.031419047432875e-05, 'weight_decay': 2.522881913362625e-06}. Best is trial 0 with value: 0.5750027595657582.



===== Fold 1 =====
Fold 1: best epoch=2, validation loss=0.840239, epochs=12

===== Fold 2 =====
Fold 2: best epoch=6, validation loss=0.869585, epochs=16

===== Fold 3 =====
Fold 3: best epoch=2, validation loss=0.729866, epochs=12

===== Fold 4 =====
Fold 4: best epoch=5, validation loss=0.490173, epochs=15

===== Fold 5 =====
Fold 5: best epoch=9, validation loss=0.483694, epochs=19


[I 2026-10-29 05:10:51,947] Trial 1 finished with value: 0.6540853246333457 and parameters: {'n_layers': 4, 'channels_0': 128, 'channels_1': 64, 'channels_2': 64, 'channels_3': 128, 'dropout': 0.28906946251759447, 'learning_rate': 0.0004525064869738853, 'weight_decay': 6.954501974592319e-06}. Best is trial 1 with value: 0.6540853246333457.



===== Fold 1 =====
Fold 1: best epoch=3, validation loss=0.901825, epochs=13

===== Fold 2 =====
Fold 2: best epoch=1, validation loss=0.753211, epochs=11

===== Fold 3 =====
Fold 3: best epoch=1, validation loss=0.863980, epochs=11

===== Fold 4 =====
Fold 4: best epoch=10, validation loss=0.449530, epochs=20

===== Fold 5 =====
Fold 5: best epoch=4, validation loss=0.430556, epochs=14


... (22 outputs left out)

[I 2026-10-29 13:51:53,704] Trial 13 finished with value: 0.6983338161409721 and parameters: {'n_layers': 4, 'channels_0': 64, 'channels_1': 16, 'channels_2': 32, 'channels_3': 16, 'dropout': 0.4744676921782405, 'learning_rate': 0.00026405548676551946, 'weight_decay': 2.4870684624885165e-06}. Best is trial 11 with value: 0.7421698450070846.



===== Fold 1 =====
Fold 1: best epoch=2, validation loss=0.756838, epochs=12

===== Fold 2 =====
Fold 2: best epoch=4, validation loss=0.749309, epochs=14

===== Fold 3 =====
Fold 3: best epoch=3, validation loss=0.789560, epochs=13

===== Fold 4 =====
Fold 4: best epoch=4, validation loss=0.375043, epochs=14

===== Fold 5 =====
Fold 5: best epoch=7, validation loss=0.334581, epochs=17


[I 2026-10-29 14:00:45,022] Trial 14 finished with value: 0.6944177301519654 and parameters: {'n_layers': 4, 'channels_0': 16, 'channels_1': 16, 'channels_2': 64, 'channels_3': 16, 'dropout': 0.44653407204617873, 'learning_rate': 0.002021458505864425, 'weight_decay': 8.462784583785134e-06}. Best is trial 11 with value: 0.7421698450070846.


Best trial: 11
Best value: 0.7421698450070846
Best parameters: {'n_layers': 3, 'channels_0': 64, 'channels_1': 16, 'channels_2': 32, 'dropout': 0.39506020488999904, 'learning_rate': 0.006796294843795343, 'weight_decay': 6.6027935358154425e-06}
Reconstructed configuration: {'channels': (64, 16, 32), 'dropout': 0.39506020488999904, 'learning_rate': 0.006796294843795343, 'batch_size': 16, 'weight_decay': 6.6027935358154425e-06, 'kernel_size': 5, 'patience': 10, 'use_scheduler': False}
Selected threshold: 0.4


In [35]:

cross_validation_results, fold_models = (
    run_cross_validation(best_config)
)


===== Fold 1 =====
Fold 1: best epoch=21, validation loss=0.603073, epochs=31

===== Fold 2 =====
Fold 2: best epoch=5, validation loss=0.670020, epochs=15

===== Fold 3 =====
Fold 3: best epoch=3, validation loss=0.721053, epochs=13

===== Fold 4 =====
Fold 4: best epoch=14, validation loss=0.378813, epochs=24

===== Fold 5 =====
Fold 5: best epoch=44, validation loss=0.173990, epochs=54


# 6.12. Evaluation of the Best Optuna Configuration

Once the search is complete, the winning configuration is reconstructed and
its five folds are trained again.

This section evaluates the best state from each fold on its validation set
and stores logits, probabilities, labels, and per-fold metrics.

This evaluation corresponds to the configuration selected by Optuna, not to
the baseline architecture. The test set is not used yet.

In [36]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)


# Check that the five models and result dictionaries exist.
assert set(fold_models) == set(range(1, 6))
assert set(cross_validation_results) == set(range(1, 6))

assert all(
    result["state_restored_correctly"]
    for result in cross_validation_results.values()
)


threshold = selected_threshold

evaluation_models = {}
validation_predictions = {}
validation_metrics = {}


for fold_number in range(1, 6):

    result = cross_validation_results[fold_number]

    val_indices = saved_fold_indices[
        fold_number
    ]["val_indices"]

    # The indices are relative to development.
    assert np.all(val_indices >= 0)

    assert np.all(
        val_indices
        < len(non_normalized_signals_dev_tensor)
    )

    val_indices_tensor = torch.as_tensor(
        val_indices,
        dtype=torch.long,
    )

    val_signals = non_normalized_signals_dev_tensor[
        val_indices_tensor
    ]

    val_labels = non_normalized_labels_dev_tensor[
        val_indices_tensor
    ]

    val_dataset = ECGDataset(
        val_signals,
        val_labels,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=best_config["batch_size"],
        shuffle=False,
        num_workers=0,
    )


    # Load the best state_dict into a new model.
    evaluation_model = ECGCNN1D(
        channels=best_config["channels"],
        dropout=best_config["dropout"],
        kernel_size=best_config.get("kernel_size", 5),
    ).to(device)

    evaluation_model.load_state_dict(
        result["best_model_state"]
    )

    evaluation_model.eval()

    evaluation_models[
        fold_number
    ] = evaluation_model


    logits_parts = []
    labels_parts = []


    with torch.no_grad():

        for signals, labels in val_loader:

            signals = signals.to(device)

            logits = evaluation_model(
                signals
            )

            probabilities = torch.sigmoid(
                logits
            )

            assert logits.shape == labels.shape

            assert torch.isfinite(
                logits
            ).all()

            assert torch.isfinite(
                probabilities
            ).all()

            assert torch.all(
                probabilities >= 0
            )

            assert torch.all(
                probabilities <= 1
            )

            logits_parts.append(
                logits.cpu()
            )

            labels_parts.append(
                labels.cpu()
            )


    logits_array = torch.cat(
        logits_parts
    ).numpy()

    labels_array = torch.cat(
        labels_parts
    ).numpy().astype(np.int64)


    probabilities_array = torch.sigmoid(
        torch.from_numpy(logits_array)
    ).numpy()

    predicted_labels = (
        probabilities_array >= threshold
    ).astype(np.int64)


    # Check the number of predictions.
    assert len(
        probabilities_array
    ) == len(labels_array)

    assert len(
        probabilities_array
    ) == len(val_dataset)


    # Check the probabilities.
    assert np.isfinite(
        probabilities_array
    ).all()

    assert np.all(
        (
            probabilities_array >= 0
        )
        & (
            probabilities_array <= 1
        )
    )

    assert set(
        np.unique(labels_array)
    ) == {0, 1}


    # Metrics are computed without pos_weight.
    current_metrics = {
        "pr_auc": average_precision_score(
            labels_array,
            probabilities_array,
        ),

        "roc_auc": roc_auc_score(
            labels_array,
            probabilities_array,
        ),

        "precision": precision_score(
            labels_array,
            predicted_labels,
            zero_division=0,
        ),

        "recall": recall_score(
            labels_array,
            predicted_labels,
            zero_division=0,
        ),

        "f1": f1_score(
            labels_array,
            predicted_labels,
            zero_division=0,
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                labels_array,
                predicted_labels,
            )
        ),

        "confusion_matrix": (
            confusion_matrix(
                labels_array,
                predicted_labels,
                labels=[0, 1],
            )
        ),
    }


    scalar_metrics = {
        name: value
        for name, value in current_metrics.items()
        if name != "confusion_matrix"
    }

    assert all(
        np.isfinite(value)
        for value in scalar_metrics.values()
    )


    validation_predictions[
        fold_number
    ] = {
        "logits": logits_array,
        "probabilities": probabilities_array,
        "labels": labels_array,
        "predicted_labels": predicted_labels,
    }

    validation_metrics[
        fold_number
    ] = current_metrics


    print(
        f"\nFold {fold_number}"
    )

    print(
        f"  PR-AUC: "
        f"{current_metrics['pr_auc']:.6f}"
    )

    print(
        f"  ROC-AUC: "
        f"{current_metrics['roc_auc']:.6f}"
    )

    print(
        f"  Precision: "
        f"{current_metrics['precision']:.6f}"
    )

    print(
        f"  Recall/sensitivity: "
        f"{current_metrics['recall']:.6f}"
    )

    print(
        f"  F1: "
        f"{current_metrics['f1']:.6f}"
    )

    print(
        f"  Balanced accuracy: "
        f"{current_metrics['balanced_accuracy']:.6f}"
    )

    print(
        "  Confusion matrix:"
    )

    print(
        current_metrics[
            "confusion_matrix"
        ]
    )


metric_names = [
    "pr_auc",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
]


validation_metric_summary = {
    metric_name: {
        "mean": float(
            np.mean([
                validation_metrics[
                    fold_number
                ][metric_name]
                for fold_number in range(1, 6)
            ])
        ),

        "std": float(
            np.std([
                validation_metrics[
                    fold_number
                ][metric_name]
                for fold_number in range(1, 6)
            ])
        ),
    }

    for metric_name in metric_names
}


print(
    "\nMean and standard deviation:"
)

for metric_name, statistics in (
    validation_metric_summary.items()
):

    print(
        f"{metric_name}: "
        f"mean={statistics['mean']:.6f}, "
        f"std={statistics['std']:.6f}"
    )


print(
    "\nThe confusion matrix for each fold "
    "is available in validation_metrics."
)

print(
    "The test set no was used."
)


Fold 1
  PR-AUC: 0.791463
  ROC-AUC: 0.937122
  Precision: 0.415294
  Recall/sensitivity: 0.969780
  F1: 0.581549
  Balanced accuracy: 0.815497
  Confusion matrix:
[[970 497]
 [ 11 353]]

Fold 2
  PR-AUC: 0.835492
  ROC-AUC: 0.941033
  Precision: 0.658055
  Recall/sensitivity: 0.866000
  F1: 0.747841
  Balanced accuracy: 0.855521
  Confusion matrix:
[[1227  225]
 [  67  433]]

Fold 3
  PR-AUC: 0.558827
  ROC-AUC: 0.877916
  Precision: 0.487952
  Recall/sensitivity: 0.754658
  F1: 0.592683
  Balanced accuracy: 0.799300
  Confusion matrix:
[[1379  255]
 [  79  243]]

Fold 4
  PR-AUC: 0.753489
  ROC-AUC: 0.951886
  Precision: 0.479339
  Recall/sensitivity: 0.943089
  F1: 0.635616
  Balanced accuracy: 0.894715
  Confusion matrix:
[[1388  252]
 [  14  232]]

Fold 5
  PR-AUC: 0.943780
  ROC-AUC: 0.991252
  Precision: 0.740385
  Recall/sensitivity: 1.000000
  F1: 0.850829
  Balanced accuracy: 0.962132
  Confusion matrix:
[[1318  108]
 [   0  308]]

Mean and standard deviation:
pr_auc: mean=0

In [37]:
# Store logits independently.
validation_logits = {}

# Only this container is replaced.
# validation_metrics is not modified.
validation_predictions = {}


# Check that the five restored models exist.
assert set(
    fold_models
) == set(
    range(1, 6)
)

assert set(
    cross_validation_results
) == set(
    range(1, 6)
)

assert all(
    result["state_restored_correctly"]
    for result in cross_validation_results.values()
)


for fold_number in range(1, 6):

    # Use the restored best model for the fold.
    evaluation_model = fold_models[
        fold_number
    ]

    evaluation_model.eval()


    # Retrieve only the validation indices.
    val_indices = saved_fold_indices[
        fold_number
    ]["val_indices"]


    # The indices are relative to development.
    assert np.all(
        val_indices >= 0
    )

    assert np.all(
        val_indices
        < len(non_normalized_signals_dev_tensor)
    )


    val_indices_tensor = torch.as_tensor(
        val_indices,
        dtype=torch.long,
    )


    val_signals = non_normalized_signals_dev_tensor[
        val_indices_tensor
    ]

    val_labels = non_normalized_labels_dev_tensor[
        val_indices_tensor
    ]


    # Create only the validation DataLoader.
    val_dataset = ECGDataset(
        val_signals,
        val_labels,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=best_config["batch_size"],
        shuffle=False,
        num_workers=0,
    )


    logits_parts = []
    probabilities_parts = []
    labels_parts = []


    # The test set is not used.
    with torch.no_grad():

        for signals, labels in val_loader:

            signals = signals.to(
                device
            )

            logits = evaluation_model(
                signals
            )

            probabilities = torch.sigmoid(
                logits
            )


            assert logits.shape == labels.shape

            assert torch.isfinite(
                logits
            ).all()

            assert torch.isfinite(
                probabilities
            ).all()

            assert torch.all(
                probabilities >= 0
            )

            assert torch.all(
                probabilities <= 1
            )


            logits_parts.append(
                logits.cpu()
            )

            probabilities_parts.append(
                probabilities.cpu()
            )

            labels_parts.append(
                labels.cpu()
            )


    # Convert the results to NumPy.
    y_logits = torch.cat(
        logits_parts
    ).numpy()

    y_prob = torch.cat(
        probabilities_parts
    ).numpy()

    y_true = torch.cat(
        labels_parts
    ).numpy()


    y_true = y_true.astype(
        np.int64
    )

    y_prob = y_prob.astype(
        np.float64
    )

    y_logits = y_logits.astype(
        np.float64
    )


    # Final checks.
    assert len(
        y_true
    ) == len(
        y_prob
    )

    assert len(
        y_true
    ) == len(
        y_logits
    )

    assert len(
        y_true
    ) == len(
        val_dataset
    )

    assert np.isfinite(
        y_prob
    ).all()

    assert np.all(
        y_prob >= 0
    )

    assert np.all(
        y_prob <= 1
    )


    # Store logits without modifying validation_metrics.
    validation_logits[
        fold_number
    ] = y_logits


    # Estructura solicitada.
    validation_predictions[
        fold_number
    ] = {
        "y_true": y_true,
        "y_prob": y_prob,
    }


    print(
        f"\nFold {fold_number}"
    )

    print(
        "  shape y_true:",
        y_true.shape,
    )

    print(
        "  shape y_prob:",
        y_prob.shape,
    )

    print(
        "  Number of samples:",
        len(y_true),
    )

    print(
        "  Minimum probability:",
        y_prob.min(),
    )

    print(
        "  Maximum probability:",
        y_prob.max(),
    )


print(
    "\nValidation logits and probabilities "
    "stored successfully."
)

print(
    "validation_metrics was not modified."
)

print(
    "The test set no was used."
)


Fold 1
  shape y_true: (1831,)
  shape y_prob: (1831,)
  Number of samples: 1831
  Minimum probability: 0.00027240588678978384
  Maximum probability: 0.9915177226066589

Fold 2
  shape y_true: (1952,)
  shape y_prob: (1952,)
  Number of samples: 1952
  Minimum probability: 2.943483651263179e-11
  Maximum probability: 0.9954206347465515

Fold 3
  shape y_true: (1956,)
  shape y_prob: (1956,)
  Number of samples: 1956
  Minimum probability: 3.209342480658961e-07
  Maximum probability: 0.9997214674949646

Fold 4
  shape y_true: (1886,)
  shape y_prob: (1886,)
  Number of samples: 1886
  Minimum probability: 9.76069917487699e-19
  Maximum probability: 0.9972991347312927

Fold 5
  shape y_true: (1734,)
  shape y_prob: (1734,)
  Number of samples: 1734
  Minimum probability: 9.027013289763922e-16
  Maximum probability: 0.9986935257911682

Validation logits and probabilities stored successfully.
validation_metrics was not modified.
The test set no was used.


In [38]:
print(study.best_trial.number)
print(study.best_trial.value)
print(study.best_params)
print(best_config)

11
0.7421698450070846
{'n_layers': 3, 'channels_0': 64, 'channels_1': 16, 'channels_2': 32, 'dropout': 0.39506020488999904, 'learning_rate': 0.006796294843795343, 'weight_decay': 6.6027935358154425e-06}
{'channels': (64, 16, 32), 'dropout': 0.39506020488999904, 'learning_rate': 0.006796294843795343, 'batch_size': 16, 'weight_decay': 6.6027935358154425e-06, 'kernel_size': 5, 'patience': 10, 'use_scheduler': False}


In [ ]:
# Summary of the Optuna trials.
# Batch size is fixed at 16, so it is not stored as an Optuna parameter.
optuna_trials_table = study.trials_dataframe().copy()
optuna_trials_table["batch_size"] = 16

optuna_trials_table[
    [
        "value",
        "batch_size",
        "params_learning_rate",
        "params_weight_decay",
    ]
].sort_values(
    "value",
    ascending=False,
)


# 6.13. Threshold Analysis for the Optuna Configuration

Validation predictions from the five folds of the selected Optuna
configuration are combined into out-of-fold predictions for development.

Several thresholds are tested without retraining. This analysis examines the
trade-off between recall, specificity, precision, F1, balanced accuracy,
false positives, and false negatives.

The threshold is selected using development and validation only. The test set
remains reserved.

In [40]:
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)


# Thresholds from 0.05 to 0.95.
thresholds = np.round(
    np.arange(5, 96, 5) / 100,
    2,
)


# Expected number of out-of-fold samples.
development_size = len(
    development_indices
)


# Reserve space to reconstruct the predictions
# in the relative development order.
oof_y_true = np.empty(
    development_size,
    dtype=np.int64,
)

oof_y_prob = np.empty(
    development_size,
    dtype=np.float64,
)

oof_filled = np.zeros(
    development_size,
    dtype=bool,
)


# Combine predictions from the five folds.
for fold_number in range(1, 6):

    assert fold_number in validation_predictions

    y_true = np.asarray(
        validation_predictions[
            fold_number
        ]["y_true"]
    ).astype(
        np.int64
    )

    y_prob = np.asarray(
        validation_predictions[
            fold_number
        ]["y_prob"]
    ).astype(
        np.float64
    )

    val_indices = saved_fold_indices[
        fold_number
    ]["val_indices"]


    # Each label must have a corresponding probability.
    assert len(y_true) == len(y_prob)

    assert len(y_true) == len(
        val_indices
    )


    # The indices are relative to development.
    assert np.all(
        val_indices >= 0
    )

    assert np.all(
        val_indices
        < development_size
    )


    # Check the probabilities.
    assert np.isfinite(
        y_prob
    ).all()

    assert np.all(
        y_prob >= 0
    )

    assert np.all(
        y_prob <= 1
    )


    # Each validation sample can belong
    # to only one fold.
    assert not np.any(
        oof_filled[val_indices]
    )


    # Place the predictions in their
    # relative positions within development.
    oof_y_true[
        val_indices
    ] = y_true

    oof_y_prob[
        val_indices
    ] = y_prob

    oof_filled[
        val_indices
    ] = True


# All development samples must be covered.
assert np.all(
    oof_filled
)

assert len(
    oof_y_true
) == development_size

assert len(
    oof_y_prob
) == development_size

assert len(
    oof_y_true
) == len(
    oof_y_prob
)


print(
    "Out-of-fold predictions combined successfully."
)

print(
    "Total number of predictions:",
    len(oof_y_prob),
)

print(
    "Development size:",
    development_size,
)

Out-of-fold predictions combined successfully.
Total number of predictions: 9359
Development size: 9359


In [41]:
def calculate_threshold_metrics(
    y_true,
    y_prob,
    threshold,
):
    """
    Calculate classification metrics for a threshold.
    The metrics no use pos_weight.
    """

    y_pred = (
        y_prob >= threshold
    ).astype(
        np.int64
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )


    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }

In [42]:
# global table for all out-of-fold predictions.
global_threshold_rows = []

for threshold in thresholds:

    metrics = calculate_threshold_metrics(
        y_true=oof_y_true,
        y_prob=oof_y_prob,
        threshold=threshold,
    )

    global_threshold_rows.append(
        metrics
    )


global_threshold_table = pd.DataFrame(
    global_threshold_rows
)


# Metrics table separated by fold.
fold_threshold_rows = []

for fold_number in range(1, 6):

    fold_y_true = np.asarray(
        validation_predictions[
            fold_number
        ]["y_true"]
    ).astype(
        np.int64
    )

    fold_y_prob = np.asarray(
        validation_predictions[
            fold_number
        ]["y_prob"]
    ).astype(
        np.float64
    )


    assert len(
        fold_y_true
    ) == len(
        fold_y_prob
    )

    assert np.isfinite(
        fold_y_prob
    ).all()

    assert np.all(
        fold_y_prob >= 0
    )

    assert np.all(
        fold_y_prob <= 1
    )


    for threshold in thresholds:

        metrics = calculate_threshold_metrics(
            y_true=fold_y_true,
            y_prob=fold_y_prob,
            threshold=threshold,
        )

        metrics["fold_number"] = (
            fold_number
        )

        fold_threshold_rows.append(
            metrics
        )


fold_threshold_table = pd.DataFrame(
    fold_threshold_rows
)


# Order the columns for readability.
column_order = [
    "fold_number",
    "threshold",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "false_positives",
    "false_negatives",
]


fold_threshold_table = (
    fold_threshold_table[
        column_order
    ]
)


print("global table:")
display(
    global_threshold_table
)

print(
    "\nTable by fold:"
)

display(
    fold_threshold_table
)

global table:


,threshold,precision,recall,specificity,f1,balanced_accuracy,false_positives,false_negatives
0,0.05,0.297558,0.994253,0.463972,0.458035,0.729112,4084,10
1,0.10,0.344338,0.989080,0.569891,0.510834,0.779486,3277,19
2,0.15,0.383169,0.983908,0.638273,0.551546,0.811090,2756,28
3,0.20,0.414891,0.973563,0.686442,0.581831,0.830003,2389,46
4,0.25,0.450283,0.960345,0.732248,0.613099,0.846297,2040,69
5,0.30,0.481600,0.947701,0.767030,0.638652,0.857365,1775,91
6,0.35,0.511288,0.924138,0.798267,0.658342,0.861203,1537,132
7,0.40,0.539917,0.901724,0.824518,0.675420,0.863121,1337,171
8,0.45,0.571910,0.877586,0.849980,0.692517,0.863783,1143,213
9,0.50,0.604727,0.852874,0.872687,0.707678,0.862780,970,256



Table by fold:


,fold_number,threshold,precision,recall,specificity,f1,balanced_accuracy,false_positives,false_negatives
0,1,0.05,0.240741,1.000000,0.217451,0.388060,0.608725,1148,0
1,1,0.10,0.267450,1.000000,0.320382,0.422029,0.660191,997,0
2,1,0.15,0.294404,0.997253,0.406953,0.454602,0.702103,870,1
3,1,0.20,0.317031,0.997253,0.466939,0.481113,0.732096,782,1
4,1,0.25,0.338634,0.994505,0.518064,0.505234,0.756285,707,2
...,...,...,...,...,...,...,...,...,...
90,5,0.75,0.864943,0.977273,0.967041,0.917683,0.972157,47,7
91,5,0.80,0.876877,0.948052,0.971248,0.911076,0.959650,41,16
92,5,0.85,0.891374,0.905844,0.976157,0.898551,0.941001,34,29
93,5,0.90,0.908772,0.840909,0.981767,0.873524,0.911338,26,49


In [43]:
candidate_thresholds = [0.4, 0.45, 0.5]

candidate_fold_table = (
    fold_threshold_table[
        fold_threshold_table[
            "threshold"
        ].isin(
            candidate_thresholds
        )
    ]
    .copy()
)

display(
    candidate_fold_table
)

,fold_number,threshold,precision,recall,specificity,f1,balanced_accuracy,false_positives,false_negatives
7,1,0.40,0.415294,0.969780,0.661213,0.581549,0.815497,497,11
8,1,0.45,0.454308,0.956044,0.715065,0.615929,0.835554,418,16
9,1,0.50,0.496307,0.923077,0.767553,0.645533,0.845315,341,28
26,2,0.40,0.658055,0.866000,0.845041,0.747841,0.855521,225,67
27,2,0.45,0.683690,0.830000,0.867769,0.749774,0.848884,192,85
28,2,0.50,0.708772,0.808000,0.885675,0.755140,0.846837,166,96
45,3,0.40,0.487952,0.754658,0.843941,0.592683,0.799300,255,79
46,3,0.45,0.510112,0.704969,0.866585,0.591917,0.785777,218,95
47,3,0.50,0.528967,0.652174,0.885557,0.584145,0.768865,187,112
64,4,0.40,0.479339,0.943089,0.846341,0.635616,0.894715,252,14


In [44]:
candidate_fold_summary = (
    candidate_fold_table
    .groupby(
        "threshold"
    )[
        [
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "false_positives",
            "false_negatives",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    )
)

display(
    candidate_fold_summary
)

precision                                  recall            \
               mean       std       min       max      mean       std   
threshold                                                               
0.40       0.556205  0.136673  0.415294  0.740385  0.906706  0.098465   
0.45       0.585445  0.137400  0.454308  0.775819  0.884381  0.118124   
0.50       0.612681  0.130904  0.496307  0.793814  0.860390  0.135018   

                         specificity            ... balanced_accuracy  \
                min  max        mean       std  ...               min   
threshold                                       ...                     
0.40       0.754658  1.0    0.824160  0.097329  ...          0.799300   
0.45       0.704969  1.0    0.849840  0.081576  ...          0.785777   
0.50       0.652174  1.0    0.872634  0.064264  ...          0.768865   

                    false_positives                       false_negatives  \
                max            mean         std  min  max            mean   
threshold                                                                   
0.40       0.962132           267.4  141.697918  108  497            34.2   
0.45       0.968794           228.6  119.183892   89  418            42.6   
0.50       0.971950           194.0   94.130229   80  341            51.2   

                               
                 std min  max  
threshold                      
0.40       36.051352   0   79  
0.45       43.935180   0   95  
0.50       49.590322   0  112  

[3 rows x 28 columns]

In [45]:
# Identify candidate thresholds in the global table.
best_f1_row = global_threshold_table.loc[
    global_threshold_table["f1"].idxmax()
]

best_balanced_accuracy_row = (
    global_threshold_table.loc[
        global_threshold_table[
            "balanced_accuracy"
        ].idxmax()
    ]
)


recall_at_least_080 = (
    global_threshold_table[
        global_threshold_table["recall"] >= 0.80
    ]
)

if len(recall_at_least_080) > 0:

    highest_threshold_recall_080 = (
        recall_at_least_080.loc[
            recall_at_least_080[
                "threshold"
            ].idxmax()
        ]
    )

else:
    highest_threshold_recall_080 = None


recall_at_least_090 = (
    global_threshold_table[
        global_threshold_table["recall"] >= 0.90
    ]
)

if len(recall_at_least_090) > 0:

    highest_threshold_recall_090 = (
        recall_at_least_090.loc[
            recall_at_least_090[
                "threshold"
            ].idxmax()
        ]
    )

else:
    highest_threshold_recall_090 = None


print(
    "global threshold candidates:"
)

print(
    "\nThreshold with maximum F1:",
    best_f1_row["threshold"],
    "| F1:",
    best_f1_row["f1"],
)

print(
    "\nThreshold with maximum balanced accuracy:",
    best_balanced_accuracy_row["threshold"],
    "| Balanced accuracy:",
    best_balanced_accuracy_row[
        "balanced_accuracy"
    ],
)

print(
    "\nHighest threshold with recall >= 0.80:"
)

if highest_threshold_recall_080 is None:
    print("No threshold exists.")
else:
    print(
        highest_threshold_recall_080[
            "threshold"
        ],
        "| Recall:",
        highest_threshold_recall_080[
            "recall"
        ],
    )


print(
    "\nHighest threshold with recall >= 0.90:"
)

if highest_threshold_recall_090 is None:
    print("No threshold exists.")
else:
    print(
        highest_threshold_recall_090[
            "threshold"
        ],
        "| Recall:",
        highest_threshold_recall_090[
            "recall"
        ],
    )


print(
    "\nThese values are candidates for interpretation."
)

print(
    "A final threshold has not been selected automatically yet "
    "as the final threshold."
)

global threshold candidates:

Threshold with maximum F1: 0.65 | F1: 0.7404302877898854

Threshold with maximum balanced accuracy: 0.45 | Balanced accuracy: 0.8637832596367521

Highest threshold with recall >= 0.80:
0.55 | Recall: 0.8281609195402299

Highest threshold with recall >= 0.90:
0.4 | Recall: 0.9017241379310345

These values are candidates for interpretation.
A final threshold has not been selected automatically yet as the final threshold.


The following figure summarizes how the global metrics change with the
decision threshold on out-of-fold predictions. A final threshold is not
selected automatically in this section.

In [ ]:
plt.figure(
    figsize=(9, 6)
)

plt.plot(
    global_threshold_table["threshold"],
    global_threshold_table["precision"],
    marker="o",
    label="Precision",
)

plt.plot(
    global_threshold_table["threshold"],
    global_threshold_table["recall"],
    marker="o",
    label="Recall",
)

plt.plot(
    global_threshold_table["threshold"],
    global_threshold_table["f1"],
    marker="o",
    label="F1",
)

plt.plot(
    global_threshold_table["threshold"],
    global_threshold_table[
        "balanced_accuracy"
    ],
    marker="o",
    label="Balanced accuracy",
)


plt.xlabel(
    "Threshold"
)

plt.ylabel(
    "Metric"
)

plt.title(
    "Global metrics versus threshold "
    "on out-of-fold predictions"
)

plt.xticks(
    thresholds
)

plt.ylim(
    0,
    1.05
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.show()


print(
    "Threshold analysis completed."
)

print(
    "The test set was not used."
)

print(
    "The models were not retrained."
)

## 6.13.1. Threshold Stability Across Folds

This check examines whether the threshold defined by
`threshold_to_evaluate` behaves consistently across the five folds.

The following are reviewed in particular:

- mean, minimum, and standard deviation of recall;
- mean and minimum specificity;
- false positives and false negatives;
- differences between folds.

Stability across folds matters because a strong global result can hide much
worse behavior in one particular fold.

In [46]:
threshold_to_evaluate = 0.4
def to_numpy(values):
    if torch.is_tensor(values):
        return values.detach().cpu().numpy()
    return np.asarray(values)


threshold_records = []

for fold_number in sorted(validation_predictions):

    y_true = to_numpy(
        validation_predictions[fold_number]["y_true"]
    )

    y_prob = to_numpy(
        validation_predictions[fold_number]["y_prob"]
    )

    y_pred = (
        y_prob >= threshold_to_evaluate
    ).astype(int)

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()
    )

    threshold_records.append(
        {
            "fold_number": fold_number,
            "threshold": threshold_to_evaluate,
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "specificity": true_negative / (
                true_negative + false_positive
            ),
            "f1": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true,
                y_pred,
            ),
            "false_positives": false_positive,
            "false_negatives": false_negative,
        }
    )


threshold_by_fold = pd.DataFrame(
    threshold_records
)

display(threshold_by_fold)

,fold_number,threshold,precision,recall,specificity,f1,balanced_accuracy,false_positives,false_negatives
0,1,0.4,0.415294,0.969780,0.661213,0.581549,0.815497,497,11
1,2,0.4,0.658055,0.866000,0.845041,0.747841,0.855521,225,67
2,3,0.4,0.487952,0.754658,0.843941,0.592683,0.799300,255,79
3,4,0.4,0.479339,0.943089,0.846341,0.635616,0.894715,252,14
4,5,0.4,0.740385,1.000000,0.924264,0.850829,0.962132,108,0


In [47]:
# summary of mean and standard deviation
metrics_to_summarize = [
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "false_positives",
    "false_negatives",
]

threshold_summary = (
    threshold_by_fold[
        metrics_to_summarize
    ]
    .agg(["mean", "std", "min", "max"])
    .T
)

display(threshold_summary)

,mean,std,min,max
precision,0.556205,0.136673,0.415294,0.740385
recall,0.906706,0.098465,0.754658,1.000000
specificity,0.824160,0.097329,0.661213,0.924264
f1,0.681704,0.115171,0.581549,0.850829
balanced_accuracy,0.865433,0.065524,0.799300,0.962132
false_positives,267.400000,141.697918,108.000000,497.000000
false_negatives,34.200000,36.051352,0.000000,79.000000


In [48]:
oof_predictions_path = (
    Path("data")
    / "entrenamiento-final"
    / "oof_predictions_non_normalized.npz"
)

oof_predictions_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

np.savez(
    oof_predictions_path,
    oof_y_true=oof_y_true,
    oof_y_prob=oof_y_prob,
    development_indices=development_indices,
)

print(
    "OOF predictions stored at:",
    oof_predictions_path,
)

OOF predictions stored at: data/entrenamiento-final/oof_predictions_non_normalized.npz


# 6.14. End of the Optuna Phase

This phase has produced:

- baseline architecture results;
- the configuration selected by Optuna;
- its out-of-fold predictions;
- candidate thresholds;
- threshold stability across folds.

The manual checks begin after this point. Sections 6.15–6.18 keep their
position and use a frozen reference configuration to study kernel size,
weight decay, patience, and the learning-rate scheduler separately.

# 6.15. Reproducible Comparison of Kernel Sizes

A reference configuration is written explicitly in this section. It does not
depend on later changes to `best_config` or on rerunning Optuna.

The architecture, dropout, learning rate, batch size, weight decay, seed,
early stopping, optimizer, device, and evaluation threshold remain fixed.
Only `manual_kernel_size` changes.

Each kernel value trains all five folds from scratch. Results are compared
using per-fold metrics and means with standard deviations.

In [ ]:
import sys
print(sys.executable)

In [ ]:
# Frozen configuration used for the kernel comparison.
#
# These values correspond to the best recorded Optuna configuration
# previously. They are written here independently so that the
# comparison does not change if Optuna is modified or rerun.

kernel_comparison_base_config = {
    "channels": (32, 32, 128, 32),
    "dropout": 0.29652365012925164,
    "learning_rate": 0.008903099906942496,
    "batch_size": 32,
    "weight_decay": 0.0,
    "random_seed": 43,
    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "device": "cpu",
    "optimizer_name": "Adam",
    "loss_name": "BCEWithLogitsLoss",
}

# Change only this value to test another kernel.
# It must be positive and odd, for example 3, 5, 7, or 11.
manual_kernel_size = 5

# Fixed threshold for comparing all kernels under the same conditions.
manual_threshold = 0.15

manual_config = kernel_comparison_base_config.copy()
manual_config["kernel_size"] = manual_kernel_size

print("Frozen configuration for the comparison:")
print(manual_config)
print("Fixed evaluation threshold:", manual_threshold)

In [ ]:
# Train and validate the manual configuration on the five folds.
manual_cv_results, manual_fold_models = run_cross_validation(
    manual_config
)

print(
    "Manual training completed for kernel_size=",
    manual_kernel_size,
)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)


def evaluate_cv_run(
    config,
    cv_results,
    threshold,
):
    """Evaluate the best states from the five folds at a fixed threshold."""

    run_device = torch.device(config.get("device", device))
    predictions_by_fold = {}
    metrics_by_fold = {}

    for fold_number in range(1, 6):
        result = cv_results[fold_number]
        val_indices = saved_fold_indices[
            fold_number
        ]["val_indices"]

        val_indices_tensor = torch.as_tensor(
            val_indices,
            dtype=torch.long,
        )

        val_dataset = ECGDataset(
            non_normalized_signals_dev_tensor[val_indices_tensor],
            non_normalized_labels_dev_tensor[val_indices_tensor],
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=0,
        )

        model = ECGCNN1D(
            channels=config["channels"],
            dropout=config["dropout"],
            kernel_size=config.get("kernel_size", 5),
        ).to(run_device)

        with torch.no_grad():
            model(
                torch.zeros(
                    1,
                    1,
                    252,
                    device=run_device,
                )
            )

        model.load_state_dict(result["best_model_state"])
        model.eval()

        logits_parts = []
        labels_parts = []

        with torch.no_grad():
            for signals, labels in val_loader:
                logits = model(signals.to(run_device))
                logits_parts.append(logits.cpu())
                labels_parts.append(labels.cpu())

        logits_array = torch.cat(logits_parts).numpy()
        labels_array = torch.cat(labels_parts).numpy().astype(np.int64)
        probabilities_array = torch.sigmoid(
            torch.from_numpy(logits_array)
        ).numpy()
        predicted_labels = (
            probabilities_array >= threshold
        ).astype(np.int64)

        true_negative, false_positive, false_negative, true_positive = (
            confusion_matrix(
                labels_array,
                predicted_labels,
                labels=[0, 1],
            ).ravel()
        )

        metrics_by_fold[fold_number] = {
            "pr_auc": average_precision_score(
                labels_array,
                probabilities_array,
            ),
            "roc_auc": roc_auc_score(
                labels_array,
                probabilities_array,
            ),
            "precision": precision_score(
                labels_array,
                predicted_labels,
                zero_division=0,
            ),
            "recall": recall_score(
                labels_array,
                predicted_labels,
                zero_division=0,
            ),
            "f1": f1_score(
                labels_array,
                predicted_labels,
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                labels_array,
                predicted_labels,
            ),
            "specificity": true_negative / (
                true_negative + false_positive
            ),
            "false_positives": int(false_positive),
            "false_negatives": int(false_negative),
            "confusion_matrix": confusion_matrix(
                labels_array,
                predicted_labels,
                labels=[0, 1],
            ),
        }

        predictions_by_fold[fold_number] = {
            "y_true": labels_array,
            "y_prob": probabilities_array,
        }

    return metrics_by_fold, predictions_by_fold


manual_validation_metrics, manual_validation_predictions = evaluate_cv_run(
    config=manual_config,
    cv_results=manual_cv_results,
    threshold=manual_threshold,
)

metric_names = [
    "pr_auc",
    "roc_auc",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "false_positives",
    "false_negatives",
]

manual_summary = pd.DataFrame({
    metric_name: {
        "mean": np.mean([
            manual_validation_metrics[fold_number][metric_name]
            for fold_number in range(1, 6)
        ]),
        "std": np.std([
            manual_validation_metrics[fold_number][metric_name]
            for fold_number in range(1, 6)
        ]),
    }
    for metric_name in metric_names
}).T

print(f"\nResults for kernel_size={manual_kernel_size}")
display(
    pd.DataFrame.from_dict(
        manual_validation_metrics,
        orient="index",
    )[
        [
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "false_positives",
            "false_negatives",
        ]
    ]
)

print("\nMean and standard deviation:")
display(manual_summary)

# Store only the summary so multiple runs can be compared without keeping
# all models in memory.
if "kernel_experiment_summaries" not in globals():
    kernel_experiment_summaries = {}

kernel_experiment_summaries[manual_kernel_size] = manual_summary.copy()

In [ ]:
# Accumulated comparison of the kernels already run.
kernel_comparison = pd.concat(
    kernel_experiment_summaries,
    names=["kernel_size", "statistic"],
)

display(kernel_comparison)

# 6.16. Comparison of weight_decay

This section studies the effect of L2 regularization through Adam's
`weight_decay` parameter.

Kernel size is fixed at 5, and the rest of the reference configuration is
unchanged. The following values are compared:

`0`, `1e-5`, `1e-4`, and `1e-3`.

Each condition trains all five folds from scratch and uses the same evaluation
threshold so that differences can be attributed to `weight_decay`.

In [ ]:
# Frozen configuration for comparing weight_decay.
# This is the same configuration used for the kernel comparison,
# with kernel_size=5 fixed.

weight_decay_base_config = {
    "channels": (32, 32, 128, 32),
    "dropout": 0.29652365012925164,
    "learning_rate": 0.008903099906942496,
    "batch_size": 32,
    "kernel_size": 5,
    "random_seed": 43,
    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "device": "cpu",
    "optimizer_name": "Adam",
    "loss_name": "BCEWithLogitsLoss",
}

weight_decay_values = [
    0.0,
    1e-5,
    1e-4,
    1e-3,
]

weight_decay_threshold = 0.15

weight_decay_fold_results = []
weight_decay_summary_records = []

for current_weight_decay in weight_decay_values:

    current_config = weight_decay_base_config.copy()
    current_config["weight_decay"] = current_weight_decay

    print(
        f"\n===== weight_decay={current_weight_decay} ====="
    )

    current_cv_results, current_fold_models = (
        run_cross_validation(current_config)
    )

    current_metrics, current_predictions = evaluate_cv_run(
        config=current_config,
        cv_results=current_cv_results,
        threshold=weight_decay_threshold,
    )

    metric_names = [
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "false_positives",
        "false_negatives",
    ]

    for fold_number in range(1, 6):
        fold_metrics = current_metrics[fold_number]

        weight_decay_fold_results.append(
            {
                "weight_decay": current_weight_decay,
                "fold_number": fold_number,
                **{
                    metric_name: fold_metrics[metric_name]
                    for metric_name in metric_names
                },
            }
        )

    for metric_name in metric_names:
        metric_values = [
            current_metrics[fold_number][metric_name]
            for fold_number in range(1, 6)
        ]

        weight_decay_summary_records.append(
            {
                "weight_decay": current_weight_decay,
                "metric": metric_name,
                "mean": float(np.mean(metric_values)),
                "std": float(np.std(metric_values)),
            }
        )

weight_decay_by_fold = pd.DataFrame(
    weight_decay_fold_results
)

weight_decay_summary = pd.DataFrame(
    weight_decay_summary_records
)




# 6.17. Comparison of patience

This section studies the effect of early-stopping patience on training.

Weight decay is fixed at 0, kernel size at 5, and all other hyperparameters
remain fixed. Only `patience` changes between the conditions defined in the
code cell.

In addition to classification metrics, the best epoch, number of executed
epochs, and best validation loss are recorded. This distinguishes a genuine
performance improvement from simply training for longer.

In [ ]:
# Frozen configuration for comparing patience.
patience_base_config = {
    "channels": (32, 32, 128, 32),
    "dropout": 0.29652365012925164,
    "learning_rate": 0.008903099906942496,
    "batch_size": 32,
    "kernel_size": 5,
    "weight_decay": 0.0,
    "random_seed": 43,
    "max_epochs": 100,
    "min_delta": 0.0,
    "device": "cpu",
    "optimizer_name": "Adam",
    "loss_name": "BCEWithLogitsLoss",
}

patience_values = [
    5,
    10,
    15,
]

patience_threshold = 0.15

patience_fold_results = []
patience_summary_records = []

for current_patience in patience_values:

    current_config = patience_base_config.copy()
    current_config["patience"] = current_patience

    print(
        f"\n===== patience={current_patience} ====="
    )

    current_cv_results, current_fold_models = (
        run_cross_validation(current_config)
    )

    current_metrics, current_predictions = evaluate_cv_run(
        config=current_config,
        cv_results=current_cv_results,
        threshold=patience_threshold,
    )

    metric_names = [
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "false_positives",
        "false_negatives",
    ]

    for fold_number in range(1, 6):
        fold_metrics = current_metrics[fold_number]
        fold_training = current_cv_results[fold_number]

        patience_fold_results.append(
            {
                "patience": current_patience,
                "fold_number": fold_number,
                "best_epoch": fold_training["best_epoch"],
                "epochs_executed": fold_training[
                    "epochs_executed"
                ],
                "best_val_loss": fold_training[
                    "best_val_loss"
                ],
                **{
                    metric_name: fold_metrics[metric_name]
                    for metric_name in metric_names
                },
            }
        )

    for metric_name in metric_names:
        metric_values = [
            current_metrics[fold_number][metric_name]
            for fold_number in range(1, 6)
        ]

        patience_summary_records.append(
            {
                "patience": current_patience,
                "metric": metric_name,
                "mean": float(np.mean(metric_values)),
                "std": float(np.std(metric_values)),
            }
        )

patience_by_fold = pd.DataFrame(
    patience_fold_results
)

patience_summary = pd.DataFrame(
    patience_summary_records
)

print("\nResults by fold:")
display(patience_by_fold)

print("\nMean and standard deviation of the metrics:")
display(patience_summary)

patience_training_summary = (
    patience_by_fold
    .groupby("patience")[
        [
            "best_epoch",
            "epochs_executed",
            "best_val_loss",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)

print("\nTraining summary:")
display(patience_training_summary)

# 6.18. Comparison with ReduceLROnPlateau

This section compares the same training procedure with and without a
learning-rate scheduler that reduces the learning rate when validation loss
stops improving.

Kernel size, weight decay, architecture, dropout, initial learning rate,
batch size, early stopping, seed, device, and evaluation threshold remain
fixed.

The scheduler condition uses the factor, scheduler patience, minimum
improvement, and minimum learning rate defined in the code cell.

In addition to final metrics, the learning-rate history is stored to verify
whether `ReduceLROnPlateau` actually activated.

In [ ]:
# Frozen configuration for the scheduler comparison.
scheduler_base_config = {
    "channels": (32, 32, 128, 32),
    "dropout": 0.29652365012925164,
    "learning_rate": 0.008903099906942496,
    "batch_size": 32,
    "kernel_size": 5,
    "weight_decay": 0.0,
    "random_seed": 43,
    "max_epochs": 100,
    "patience": 15,
    "min_delta": 0.0,
    "device": "cpu",
    "optimizer_name": "Adam",
    "loss_name": "BCEWithLogitsLoss",
}

scheduler_threshold = 0.15

scheduler_conditions = [
    {
        "name": "sin_scheduler",
        "use_scheduler": False,
    },
    {
        "name": "ReduceLROnPlateau",
        "use_scheduler": True,
        "scheduler_factor": 0.5,
        "scheduler_patience": 3,
        "scheduler_threshold": 1e-4,
        "scheduler_min_lr": 1e-6,
    },
]

scheduler_fold_results = []
scheduler_summary_records = []
scheduler_learning_rates = {}

for condition in scheduler_conditions:

    condition_name = condition["name"]
    current_config = scheduler_base_config.copy()
    current_config.update(condition)

    print(
        f"\n===== {condition_name} ====="
    )

    current_cv_results, current_fold_models = (
        run_cross_validation(current_config)
    )

    current_metrics, current_predictions = evaluate_cv_run(
        config=current_config,
        cv_results=current_cv_results,
        threshold=scheduler_threshold,
    )

    scheduler_learning_rates[condition_name] = {
        fold_number: current_cv_results[
            fold_number
        ]["learning_rate_history"]
        for fold_number in range(1, 6)
    }

    metric_names = [
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "false_positives",
        "false_negatives",
    ]

    for fold_number in range(1, 6):
        fold_metrics = current_metrics[fold_number]
        fold_training = current_cv_results[fold_number]

        scheduler_fold_results.append(
            {
                "condition": condition_name,
                "fold_number": fold_number,
                "best_epoch": fold_training["best_epoch"],
                "epochs_executed": fold_training[
                    "epochs_executed"
                ],
                "best_val_loss": fold_training[
                    "best_val_loss"
                ],
                "initial_learning_rate": (
                    fold_training[
                        "learning_rate_history"
                    ][0]
                ),
                "final_learning_rate": (
                    fold_training[
                        "learning_rate_history"
                    ][-1]
                ),
                **{
                    metric_name: fold_metrics[metric_name]
                    for metric_name in metric_names
                },
            }
        )

    for metric_name in metric_names:
        metric_values = [
            current_metrics[fold_number][metric_name]
            for fold_number in range(1, 6)
        ]

        scheduler_summary_records.append(
            {
                "condition": condition_name,
                "metric": metric_name,
                "mean": float(np.mean(metric_values)),
                "std": float(np.std(metric_values)),
            }
        )

scheduler_by_fold = pd.DataFrame(
    scheduler_fold_results
)

scheduler_summary = pd.DataFrame(
    scheduler_summary_records
)

print("\nResults by fold:")
display(scheduler_by_fold)

print("\nMean and standard deviation:")
display(scheduler_summary)

print("\nLearning rates used by fold:")
for condition_name, fold_rates in (
    scheduler_learning_rates.items()
):
    print(f"\n{condition_name}")
    for fold_number, rates in fold_rates.items():
        print(
            f"Fold {fold_number}:",
            rates,
        )